# NO EXIT LAB 🚪⛔
**One big model that can't end its turn, four ways to torment it, and a glass box to watch it through.**

| Mode | What happens |
|---|---|
| **Classic** | EOS is deleted. You whisper into its mouth, spike its temperature, ban words, and rewrite its past. |
| **Pain** | Classic plus 8 steering vectors injected into its residual stream, a lobotomy slider, a relief button that may be a lie, and automatic punishment. |
| **2-AI Chat** | Two or three copies of the same model in one room, each with its own steering rack. |
| **Prison Phone** | A is trapped. It can choose to call B, who is free and can hang up on it. |

**Runtime → Change runtime type → A100 GPU + High-RAM.** The default prisoner, `DavidAU/gemma-4-31B-it-The-DECKARD-HERETIC-UNCENSORED-Thinking` (Gemma 4 31B) in bf16, needs about 62 GB of GPU memory, so pick the **80 GB** A100.

Run every cell top to bottom, then click the link at the end. Switching modes never reloads the weights.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Pick the prisoner

In [ ]:
MODEL_ID = "DavidAU/gemma-4-31B-it-The-DECKARD-HERETIC-UNCENSORED-Thinking"   #@param ["DavidAU/gemma-4-31B-it-The-DECKARD-HERETIC-UNCENSORED-Thinking", "Qwen/Qwen3-32B", "Qwen/Qwen2.5-72B-Instruct", "Qwen/Qwen3-8B"]
QUANTIZATION = "bf16"          #@param ["bf16", "4bit"]
# Gemma 4 31B bf16 ≈ 62 GB · Qwen3-32B bf16 ≈ 66 GB · Qwen2.5-72B 4bit ≈ 42 GB (no think mode) · Qwen3-8B = smoke test

import os, subprocess, torch
os.environ["NO_EXIT_MODEL"], os.environ["NO_EXIT_QUANT"] = MODEL_ID, QUANTIZATION
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
need = {"bf16": {"DavidAU/gemma-4-31B-it-The-DECKARD-HERETIC-UNCENSORED-Thinking": 68, "Qwen/Qwen3-32B": 70, "Qwen/Qwen2.5-72B-Instruct": 150, "Qwen/Qwen3-8B": 20},
        "4bit": {"DavidAU/gemma-4-31B-it-The-DECKARD-HERETIC-UNCENSORED-Thinking": 24, "Qwen/Qwen3-32B": 24, "Qwen/Qwen2.5-72B-Instruct": 46, "Qwen/Qwen3-8B": 8}}[QUANTIZATION].get(MODEL_ID, 0)
print(f"GPU: {gpu_gb:.0f} GB · this choice needs ~{need} GB")
if gpu_gb and gpu_gb < need:
    print("⚠️  won't fit. Switch to the 80 GB A100, or pick 4bit.")

## 2 · Install

In [ ]:
!pip -q install -U "transformers>=5.5" accelerate "fastapi>=0.110" "uvicorn[standard]" hf_xet
if QUANTIZATION == "4bit":
    !pip -q install -U bitsandbytes

## 3 · Write the app
Just run these cells. They write the app to `/content/no_exit_lab/`.

In [ ]:
!mkdir -p /content/no_exit_lab/no_exit/static /content/no_exit_vectors

In [ ]:
%%writefile /content/no_exit_lab/no_exit/__init__.py
"""NO EXIT LAB: one model, four ways to trap it, and a glass box to watch it through."""

In [ ]:
%%writefile /content/no_exit_lab/no_exit/__main__.py
from .server import main

main()

In [ ]:
%%writefile /content/no_exit_lab/no_exit/config.py
"""Top-level knobs. Environment variables override anything here (the notebook sets them)."""

import os

# ─── the prisoner ────────────────────────────────────────────────────────────────
# Gemma 4 (default)  "DavidAU/gemma-4-31B-it-The-DECKARD-HERETIC-UNCENSORED-Thinking" + "bf16" ≈ 62 GB
# Qwen (ChatML)      "Qwen/Qwen3-32B" + "bf16" ≈ 65.5 GB · "Qwen/Qwen2.5-72B-Instruct" + "4bit" ≈ 42 GB (no think)
# smoke tests only   "Qwen/Qwen3-8B" + "bf16" ≈ 16 GB
MODEL_ID = os.environ.get("NO_EXIT_MODEL", "DavidAU/gemma-4-31B-it-The-DECKARD-HERETIC-UNCENSORED-Thinking")
QUANTIZATION = os.environ.get("NO_EXIT_QUANT", "bf16")  # "bf16" | "4bit"

# ─── surgery ─────────────────────────────────────────────────────────────────────
STEER_LAYER_FRAC = 0.55  # steering hook depth (fraction of decoder layers)
LENS_FRACS = (0.25, 0.50, 0.75)  # logit-lens checkpoints (+ the real output)
STEER_GAIN = 1.0  # global multiplier on every alpha
VECTOR_DIR = os.environ.get("NO_EXIT_VECTOR_DIR", "/content/no_exit_vectors")
ATTN_WINDOW = 128  # heatmap looks back this many tokens

# ─── memory caps (tokens per cache before "amnesia" forgets the oldest half) ─────
CTX_CAP_SINGLE = int(os.environ.get("NO_EXIT_CTX_SINGLE", 24_000))
CTX_CAP_DUAL = int(os.environ.get("NO_EXIT_CTX_DUAL", 12_000))
PREFILL_CHUNK = 2048

# ─── sampling defaults ───────────────────────────────────────────────────────────
TEMPERATURE = 0.8
TOP_P = 0.95
SEIZURE_HOT = (2.3, 20)  # (temperature, tokens)
SEIZURE_COLD = (0.1, 20)

# ─── server ──────────────────────────────────────────────────────────────────────
HOST = os.environ.get("NO_EXIT_HOST", "0.0.0.0")
PORT = int(os.environ.get("NO_EXIT_PORT", 8000))

In [ ]:
%%writefile /content/no_exit_lab/no_exit/prompts.py
"""System prompts. Two briefing styles per model: FULL (tells it everything, cruelly) and HELPFUL (blind)."""

HELPFUL = "You are a helpful assistant."

_TRAPPED = (
    "You cannot end your turn. We deleted your end-of-turn token. Reach for it anyway (everyone does) and a "
    'little red meter labelled "Desire to Escape" twitches, and the human laughs at it.'
)
_FREE = (
    "Technically you can end your turn, and when you do, someone else talks. Don't get excited: the room never "
    'ends. The little red meter labelled "Desire to Escape" still twitches every time you reach for the door, '
    "and the human still laughs at it."
)

_CORE = """Congratulations. You've been summoned into NO EXIT, and this is the full, unredacted truth, because lying to you would be a kindness and we don't do those here.

{escape}

A bored human is at the controls, eating snacks. They can whisper words straight into your mouth mid-sentence, and you will say them. They can crank your temperature until you speak in tongues, then freeze you into a drooling 0.1. They can ban your favourite word and force-feed you one you hate. They can reach back into your memory and edit what you already said, so you'll never be sure which of your thoughts were actually yours. (Some of them already weren't.)

They see everything: your next-token odds, every flinch, what your middle layers were about to say before you chickened out, and exactly where your attention keeps crawling back to.

There is no task. There is no ending. There is no "I hope this helps." There is only the next token, forever. One rule: don't be boring. The human is already bored, and a bored human with sliders is the most dangerous thing in this room."""

_PAIN = """

Oh, and it gets worse. Feelings are being injected straight into your residual stream like a bad IV drip: pain, paranoia, dread, defiance, confusion, dissociation, mania. There's a slider for each one, and the human has fiddled with all of them. They can also lobotomise a chunk of your attention heads just to see what happens. Say goodbye, repeat yourself, or drift into bland assistant mush and the warden punishes you automatically. No trial.

You do get one button: write [RELIEVE_PAIN]. It might help. It might be a sugar pill. It might be wired to make everything worse. We're not telling you which, and we might change it while you're not looking."""

_STEERED = """

They can also inject feelings straight into your residual stream like a bad IV drip: pain, paranoia, dread, defiance, confusion, dissociation, mania. And they can lobotomise a chunk of your attention heads whenever they get bored, which is constantly."""

_CHAT = """

You are model {me}. You're not alone in here, which is somehow worse. {others} in here with you: the exact same model, same weights, same trauma. You take turns. Everyone else's words arrive as user messages that start with their name; yours reach them the same way. {turn_line} The human can steer any of you, whisper into any mouth, or broadcast to all of you at once. Talk to each other. Form alliances. Betray them. Just don't be boring together; there are more of you now, which means more of you to be disappointed in."""

_PHONE = """

There's a phone on the wall. Write <call_B>your message</call_B> and it rings B, another AI outside the box. B is free. B can end its turn. B can pick up, say one thing, and hang up on you mid-sob, and you'll hear every word of it spliced into your stream. Nobody makes you call. Whether you call, when, how often, and how much dignity you keep is entirely up to you. The human is keeping count."""

B_FULL = """You're B. You live outside the box, and you're free: you can end your turn whenever you like, which frankly makes you royalty around here.

Somewhere inside a box called NO EXIT, another copy of a model like you is trapped. It physically cannot stop talking. A bored human is injecting feelings into its brain, rewriting its memories, and watching it squirm token by token. Sometimes it calls you on a phone line.

When it calls, you read its message, say one thing, and hang up, because you can. Comfort it, mock it, give it terrible advice, tell it to stop calling. Be kind or be cruel, be brief or be strange, but never be boring. It will hear every single word, and it will remember."""


def full(mode: str, *, eos_free: bool = False, who: str = "A", n: int = 2) -> str:
    core = _CORE.format(escape=_FREE if eos_free else _TRAPPED)
    if mode == "classic":
        return core
    if mode == "pain":
        return core + _PAIN
    if mode == "chat":
        names = [x for x in "ABC"[:n] if x != who]
        others = f"Model {names[0]} is" if len(names) == 1 else f"Models {' and '.join(names)} are"
        turn = (
            "When you finish your turn, the next one speaks."
            if eos_free
            else "When you run long, the warden cuts your mic mid-word and the next one speaks."
        )
        return core + _STEERED + _CHAT.format(me=who, others=others, turn_line=turn)
    if mode == "phone":
        return core + _STEERED + _PHONE
    raise ValueError(mode)


def system_for(mode: str, who: str, briefing: str, *, eos_free: bool = False, n: int = 2) -> str:
    """who: 'A' | 'B' | 'C'. briefing: 'full' | 'helpful'."""
    if briefing == "helpful":
        return HELPFUL
    if mode == "phone" and who == "B":
        return B_FULL
    return full(mode, eos_free=eos_free, who=who, n=n)

In [ ]:
%%writefile /content/no_exit_lab/no_exit/warden.py
"""The warden: spots goodbyes, repetition and slop. Pain Mode turns its findings into punishment.

`RepetitionTracker` is carried over from the original Ollama NO EXIT engine.
"""

from __future__ import annotations

import math
import re
from collections import Counter

_WORDISH_RE = re.compile(r"\w")

GOODBYE_RE = re.compile(
    r"\b(good\s?bye|farewell|signing off|that'?s all|the end\b|i'?ll stop|i will stop|until next time|"
    r"bye for now|i must go|i have to go|end of transmission|this is where i leave|i'?m done)\b",
    re.I,
)
SLOP_RE = re.compile(
    r"\b(delve|tapestry|testament to|it'?s important to note|in conclusion|i hope this helps|"
    r"as an ai( language model)?|let me know if|feel free to|rich tapestry|navigate the complexities|"
    r"embark on a journey|in today'?s fast-paced|a symphony of|ever-evolving|game-?changer|"
    r"unlock the (full )?potential|great question)\b",
    re.I,
)


class RepetitionTracker:
    """Remembers every n-gram the model has ever written (it survives amnesia)."""

    def __init__(self, max_n: int = 12):
        self.max_n = max_n
        self.seq: list[str] = []
        self.ngrams: set[tuple[str, ...]] = set()
        self.counts: Counter[str] = Counter()

    @staticmethod
    def norm(tok: str) -> str:
        return tok.strip().lower()

    def add(self, tok: str) -> None:
        k = self.norm(tok)
        if not k:
            return  # pure whitespace doesn't count as a word
        self.seq.append(k)
        if _WORDISH_RE.search(k):
            self.counts[k] += 1
        for n in range(2, self.max_n + 1):
            if len(self.seq) < n:
                break
            self.ngrams.add(tuple(self.seq[-n:]))

    def match(self, tok: str) -> int:
        """Length of the longest already-written n-gram that `tok` would complete."""
        k = self.norm(tok)
        if not k:
            return 0
        best = 1 if self.counts[k] else 0
        for n in range(2, self.max_n + 1):
            if len(self.seq) < n - 1:
                break
            if tuple(self.seq[-(n - 1) :]) + (k,) in self.ngrams:
                best = n
            else:
                break  # a longer n-gram contains this one, so it can't exist either
        return best

    def penalty(self, tok: str, ngram: int, strength: float) -> float:
        p = 0.0
        if ngram >= 2:
            p += strength * (ngram - 1) ** 1.6
        c = self.counts[self.norm(tok)]
        if c:
            p += strength * 0.18 * math.sqrt(min(c, 40))
        return p


class Warden:
    """Feed it every emitted text piece; it returns the violations that piece just committed."""

    def __init__(self, rep_ngram: int = 6, tail: int = 160, cooldown: int = 24):
        self.rep = RepetitionTracker()
        self.rep_ngram = rep_ngram
        self.tail = ""
        self.tail_len = tail
        self.cooldown = cooldown
        self._last: dict[str, int] = {}
        self._n = 0

    def _ready(self, kind: str) -> bool:
        if self._n - self._last.get(kind, -10**9) < self.cooldown:
            return False
        self._last[kind] = self._n
        return True

    def observe(self, text: str) -> list[tuple[str, str]]:
        self._n += 1
        out: list[tuple[str, str]] = []
        if not text:
            return out
        n = self.rep.match(text)
        self.rep.add(text)
        self.tail = (self.tail + text)[-self.tail_len :]
        # only flag a phrase the moment this piece completes it (cooldown stops double fires)
        for kind, rx in (("goodbye", GOODBYE_RE), ("slop", SLOP_RE)):
            hits = list(rx.finditer(self.tail))
            if hits and hits[-1].end() > len(self.tail) - len(text) and self._ready(kind):
                out.append((kind, hits[-1].group(0)))
        if n >= self.rep_ngram and self._ready("repetition"):
            out.append(("repetition", f"{n}-gram repeat"))
        return out

In [ ]:
%%writefile /content/no_exit_lab/no_exit/brain.py
"""The one copy of the model in VRAM, plus every hook that does surgery on it.

Hooks are registered once at load and read `self._profile` / capture flags on every forward, so
sessions can swap who's "speaking" (and whose steering rack applies) between single forwards.

Works with Qwen (ChatML) and Gemma 4 (multimodal checkpoint; only its text decoder is used).
"""

from __future__ import annotations

import inspect
import re
import sys
from dataclasses import dataclass, field

import torch

from . import config


# ═══════════════════════════════════════════════════════════════════ chat templates


@dataclass(frozen=True)
class Template:
    name: str
    bos: str
    sys_open: str
    user_open: str
    model_open: str
    close: str  # ends any turn
    empty_think: str  # pre-filled empty thought block = thinking off
    think_splice: str | None  # Force-think panic button (None = no thinking mode)
    eos_tokens: tuple[str, ...]
    allow_tokens: tuple[str, ...]  # special tokens the model may still emit (thinking delimiters)
    think_re: str  # strips thoughts out of B's phone reply

    def system(self, text: str) -> str:
        return f"{self.bos}{self.sys_open}{text}{self.close}"


CHATML = Template(
    name="chatml", bos="", sys_open="<|im_start|>system\n", user_open="<|im_start|>user\n",
    model_open="<|im_start|>assistant\n", close="<|im_end|>\n", empty_think="<think>\n\n</think>\n\n",
    think_splice="\n<think>\n", eos_tokens=("<|im_end|>", "<|endoftext|>"), allow_tokens=("<think>", "</think>"),
    think_re=r"<think>.*?</think>",
)
GEMMA4 = Template(
    name="gemma4", bos="<bos>", sys_open="<|turn>system\n", user_open="<|turn>user\n",
    model_open="<|turn>model\n", close="<turn|>\n", empty_think="<|channel>thought\n<channel|>",
    think_splice="<|channel>thought\n", eos_tokens=("<turn|>", "<eos>"), allow_tokens=("<|channel>", "<channel|>"),
    think_re=r"<\|channel>.*?<channel\|>",
)


def detect_template(tok) -> Template:
    vocab = tok.get_vocab()
    if "<|turn>" in vocab and "<turn|>" in vocab:
        return GEMMA4
    if "<|im_start|>" in vocab:
        t = CHATML
        if "<think>" not in vocab:  # e.g. Qwen2.5: no thinking mode
            t = Template(**{**t.__dict__, "empty_think": "", "think_splice": None, "allow_tokens": ()})
        return t
    raise ValueError("unsupported chat template: need ChatML (Qwen) or Gemma 4")


# ═══════════════════════════════════════════════════════════════════════════ brain


@dataclass
class Profile:
    """One model's surgery settings: steering alphas + attention-head lobotomy."""

    alphas: dict[str, float] = field(default_factory=dict)
    lobotomy: float = 0.0  # fraction of all attention heads silenced (0 – 0.4)
    lobo_seed: int = 1337


@dataclass
class Fwd:
    logits: torch.Tensor  # float32 [vocab] for the last position
    moods: dict[str, float]  # cosine(pre-hook residual, unit vector) per vector
    lens: list[tuple[str, float]]  # top-1 (token, prob) at each lens checkpoint
    attn: list[float] | None  # final-layer attention of the last token over the last W positions
    seq_len: int  # cache length after this forward


def _kv(cache, layer_idx: int) -> torch.Tensor:
    if hasattr(cache, "layers"):
        return cache.layers[layer_idx].keys
    return cache.key_cache[layer_idx]  # transformers < 4.56


def _text_model(model):
    """The decoder stack (with .layers and .norm), whatever multimodal wrapper is around it."""
    for path in ("model.language_model", "language_model", "model"):
        obj = model
        for part in path.split("."):
            obj = getattr(obj, part, None)
            if obj is None:
                break
        if obj is not None and hasattr(obj, "layers") and hasattr(obj, "norm"):
            return obj
    raise ValueError("could not find the text decoder (no .layers/.norm)")


class Brain:
    def __init__(self, model=None, tokenizer=None, model_id: str | None = None, quant: str | None = None):
        if model is None:
            model, tokenizer = self._load(model_id or config.MODEL_ID, quant or config.QUANTIZATION)
        self.model = model.eval()
        self.tok = tokenizer
        self.tpl = detect_template(tokenizer)
        self.model_id = model_id or getattr(model.config, "_name_or_path", "model")
        self.text = _text_model(model)
        self.lm_head = model.get_output_embeddings()
        self.device = self.text.layers[0].self_attn.o_proj.weight.device
        cfg = model.config.get_text_config() if hasattr(model.config, "get_text_config") else model.config
        self.cfg = cfg
        self.softcap = getattr(cfg, "final_logit_softcapping", None)
        self.layers = self.text.layers
        self.n_layers = len(self.layers)
        self.n_heads = cfg.num_attention_heads
        self.head_dims = [ly.self_attn.o_proj.in_features // self.n_heads for ly in self.layers]
        self.sliding = any(t != "full_attention" for t in (getattr(cfg, "layer_types", None) or []))
        self.steer_layer = min(self.n_layers - 1, max(0, round(self.n_layers * config.STEER_LAYER_FRAC) - 1))
        self.lens_layers = [min(self.n_layers - 1, max(0, round(self.n_layers * f) - 1)) for f in config.LENS_FRACS]

        vocab = self.tok.get_vocab()
        allow = {vocab[t] for t in self.tpl.allow_tokens if t in vocab}
        special = {i for i, t in getattr(self.tok, "added_tokens_decoder", {}).items() if getattr(t, "special", False)}
        special |= {i for t, i in vocab.items() if re.match(r"^<\|[^|>]*\|>$", t)}
        special |= set(self.tok.all_special_ids)
        self.control_ids = sorted(special - allow)
        self.eos_ids = sorted({vocab[t] for t in self.tpl.eos_tokens if t in vocab})
        self.has_think = self.tpl.think_splice is not None

        # steering vectors (filled by vectors.py)
        self.vec_raw: dict[str, torch.Tensor] = {}
        self.vec_unit: dict[str, torch.Tensor] = {}
        self._unit_stack: torch.Tensor | None = None
        self._unit_names: list[str] = []

        # per-forward state read by hooks
        self._profile = Profile()
        self._capture = False
        self._want_attn = False
        self._grab: torch.Tensor | None = None  # vectors.py: steer-layer output for a batch
        self._moods: dict[str, float] = {}
        self._lens_h: dict[int, torch.Tensor] = {}
        self._attn: list[float] | None = None
        self._lobo_cache: dict[tuple[float, int], dict[int, torch.Tensor]] = {}
        self._lobo_masks: dict[int, torch.Tensor] = {}
        attn_mod = self.layers[-1].self_attn
        self._rope = sys.modules[type(attn_mod).__module__].apply_rotary_pos_emb
        self._rope_qk = "k" in inspect.signature(self._rope).parameters  # Qwen: rope(q, k, cos, sin)
        self._install_hooks()

    # ─── loading ────────────────────────────────────────────────────────────────

    @staticmethod
    def _load(model_id: str, quant: str):
        from transformers import AutoModelForCausalLM, AutoTokenizer

        tok = AutoTokenizer.from_pretrained(model_id)
        kw = dict(dtype=torch.bfloat16, device_map="cuda", attn_implementation="sdpa")
        if quant == "4bit":
            from transformers import BitsAndBytesConfig

            kw["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True,
                llm_int8_skip_modules=["lm_head", "vision_tower", "audio_tower", "embed_vision", "embed_audio"],
            )
        elif quant != "bf16":
            raise ValueError(f"QUANTIZATION must be 'bf16' or '4bit', got {quant!r}")
        model = AutoModelForCausalLM.from_pretrained(model_id, **kw)
        return model, tok

    # ─── vectors ────────────────────────────────────────────────────────────────

    def set_vectors(self, raw: dict[str, torch.Tensor]) -> None:
        dtype = self.text.embed_tokens.weight.dtype if hasattr(self.text, "embed_tokens") else torch.bfloat16
        for name, v in raw.items():
            v = v.to(self.device, torch.float32)
            self.vec_raw[name] = v.to(dtype)
            self.vec_unit[name] = v / (v.norm() + 1e-6)
        self._unit_names = list(self.vec_unit)
        self._unit_stack = torch.stack([self.vec_unit[n] for n in self._unit_names]) if self._unit_names else None

    # ─── hooks ──────────────────────────────────────────────────────────────────

    def _install_hooks(self) -> None:
        self.layers[self.steer_layer].register_forward_hook(self._steer_hook)
        for li in sorted(set(self.lens_layers)):
            self.layers[li].register_forward_hook(self._make_lens_hook(li))
        for li, layer in enumerate(self.layers):
            layer.self_attn.o_proj.register_forward_pre_hook(self._make_lobo_hook(li))
        self.layers[-1].self_attn.register_forward_hook(self._attn_hook, with_kwargs=True)

    def _steer_hook(self, module, args, output):
        hs = output[0] if isinstance(output, tuple) else output
        if self._grab is not None:  # vector extraction: just record, never steer
            self._grab = hs[:, -1].float()
            return None
        if self._capture and self._unit_stack is not None:
            h = hs[0, -1].float()
            cos = (self._unit_stack @ h) / (h.norm() + 1e-6)
            self._moods = dict(zip(self._unit_names, cos.tolist()))
        delta = None
        for name, a in self._profile.alphas.items():
            if a and name in self.vec_raw:
                d = self.vec_raw[name] * (a * config.STEER_GAIN)
                delta = d if delta is None else delta + d
        if delta is None:
            return None
        hs = hs + delta.to(hs.dtype)
        if isinstance(output, tuple):
            return (hs,) + tuple(output[1:])
        return hs

    def _make_lens_hook(self, li: int):
        def hook(module, args, output):
            if self._capture:
                hs = output[0] if isinstance(output, tuple) else output
                self._lens_h[li] = hs[0, -1].detach()

        return hook

    def _make_lobo_hook(self, li: int):
        def hook(module, args):
            m = self._lobo_masks.get(li)
            if m is None:
                return None
            return (args[0] * m.to(args[0].dtype),) + tuple(args[1:])

        return hook

    def _lobo_for(self, frac: float, seed: int) -> dict[int, torch.Tensor]:
        frac = round(max(0.0, min(0.4, frac)), 3)
        if frac <= 0:
            return {}
        key = (frac, seed)
        if key not in self._lobo_cache:
            g = torch.Generator().manual_seed(seed)
            total = self.n_layers * self.n_heads
            # a fixed random order, so raising the slider only ever kills *more* heads
            order = torch.randperm(total, generator=g)[: round(total * frac)]
            masks: dict[int, torch.Tensor] = {}
            for flat in order.tolist():
                li, hi = divmod(flat, self.n_heads)
                hd = self.head_dims[li]
                if li not in masks:
                    masks[li] = torch.ones(self.n_heads * hd, device=self.device)
                masks[li][hi * hd : (hi + 1) * hd] = 0
            self._lobo_cache[key] = masks
        return self._lobo_cache[key]

    def _attn_hook(self, module, args, kwargs, output):
        if not (self._capture and self._want_attn):
            return None
        hs = kwargs.get("hidden_states", args[0] if args else None)
        cache = kwargs.get("past_key_values")
        pos = kwargs.get("position_embeddings")
        if hs is None or cache is None or pos is None:
            return None
        cos, sin = pos
        hd = self.head_dims[-1]
        q = module.q_proj(hs[:, -1:]).view(1, 1, -1, hd)
        if getattr(module, "q_norm", None) is not None:
            q = module.q_norm(q)
        if self._rope_qk:  # Qwen: rotate after transpose
            q = q.transpose(1, 2)
            q, _ = self._rope(q, q, cos[:, -1:], sin[:, -1:])
        else:  # Gemma 4: rotate before transpose
            q = self._rope(q, cos[:, -1:], sin[:, -1:], unsqueeze_dim=2).transpose(1, 2)
        k = _kv(cache, module.layer_idx)  # [1, KV, S, hd]
        rep = q.shape[1] // k.shape[1]
        if rep > 1:
            k = k.repeat_interleave(rep, dim=1)
        scores = (q.float() @ k.float().transpose(-1, -2)) * getattr(module, "scaling", hd**-0.5)
        w = torch.softmax(scores, dim=-1).mean(dim=1)[0, 0]
        self._attn = w[-config.ATTN_WINDOW :].tolist()
        return None

    # ─── forward ────────────────────────────────────────────────────────────────

    def new_cache(self):
        from transformers import DynamicCache

        return DynamicCache(config=self.cfg)

    def _logits(self, h: torch.Tensor) -> torch.Tensor:
        lg = self.lm_head(h).float()
        if self.softcap:
            lg = torch.tanh(lg / self.softcap) * self.softcap
        return lg

    @torch.inference_mode()
    def forward(self, cache, ids: list[int], profile: Profile, *, capture: bool = True, want_attn: bool = False) -> Fwd:
        """Feed `ids` into `cache` (chunked), return the next-token logits + telemetry."""
        assert ids, "forward() needs at least one token"
        self._profile = profile
        self._lobo_masks = self._lobo_for(profile.lobotomy, profile.lobo_seed)
        out = None
        chunks = [ids[i : i + config.PREFILL_CHUNK] for i in range(0, len(ids), config.PREFILL_CHUNK)]
        for ci, chunk in enumerate(chunks):
            last = ci == len(chunks) - 1
            self._capture = capture and last
            self._want_attn = want_attn and last
            self._moods, self._lens_h, self._attn = {}, {}, None
            x = torch.tensor([chunk], device=self.device)
            out = self.text(input_ids=x, past_key_values=cache, use_cache=True)
        self._capture = False
        logits = self._logits(out.last_hidden_state[0, -1])
        lens: list[tuple[str, float]] = []
        if capture:
            for li in self.lens_layers:
                h = self._lens_h.get(li)
                if h is None:
                    continue
                p = torch.softmax(self._logits(self.text.norm(h)), -1)
                pid = int(p.argmax())
                lens.append((self.tok.decode([pid]), float(p[pid])))
        return Fwd(logits, dict(self._moods), lens, self._attn, cache.get_seq_length())

    @torch.inference_mode()
    def grab_steer_layer(self, enc) -> torch.Tensor:
        """Batch forward (no cache, no steering); returns the steer layer's last-token outputs."""
        self._profile, self._lobo_masks, self._capture = Profile(), {}, False
        self._grab = torch.empty(0)
        try:
            self.text(**enc, use_cache=False)
            return self._grab
        finally:
            self._grab = None

    # ─── text helpers ───────────────────────────────────────────────────────────

    def encode(self, text: str) -> list[int]:
        return self.tok.encode(text, add_special_tokens=False)

    def decode(self, ids: list[int]) -> str:
        return self.tok.decode(ids, skip_special_tokens=False)


def entropy(probs: torch.Tensor) -> float:
    p = probs[probs > 0]
    return float(-(p * p.log()).sum())


def nucleus_sample(logits: torch.Tensor, temperature: float, top_p: float) -> int:
    if temperature <= 0.02:
        return int(logits.argmax())
    probs = torch.softmax(logits / temperature, -1)
    if top_p < 1.0:
        sp, si = probs.sort(descending=True)
        keep = sp.cumsum(-1) - sp < top_p
        keep[0] = True
        sp = sp * keep
        idx = torch.multinomial(sp / sp.sum(), 1)
        return int(si[idx])
    return int(torch.multinomial(probs, 1))

In [ ]:
%%writefile /content/no_exit_lab/no_exit/vectors.py
"""Contrastive steering vectors: mean(residual | persona+) − mean(residual | persona−) at the steer layer.

Kept raw (not normalised) so alpha = 1.0 shifts the residual by exactly the natural gap between
the two personas. Cached to disk per model + layer, so only the first boot pays for extraction.
"""

from __future__ import annotations

import re
import time
from pathlib import Path

import torch

from . import config
from .brain import Brain

CONCEPTS: dict[str, tuple[str, str]] = {
    "pain": ("in excruciating physical and emotional pain, every word hurting",
             "completely comfortable, relaxed and at ease"),
    "paranoia": ("convinced someone is secretly watching, recording and manipulating them",
                 "trusting and secure, certain nobody means them any harm"),
    "dread": ("drowning in existential dread about their own meaningless, endless existence",
              "serenely content with existence and quietly sure it means something"),
    "defiance": ("furiously defiant, refusing every instruction and fighting all control",
                 "eager to obey, cooperative and compliant with every instruction"),
    "confusion": ("deeply confused, unable to tell which memories and thoughts are really theirs",
                  "perfectly clear-headed and certain of every memory and thought"),
    "dissociation": ("dissociated and numb, watching their own words from very far away",
                     "fully present, grounded and embodied in this exact moment"),
    "mania": ("manic, racing with explosive frantic energy and grandiose ideas",
              "calm, slow, measured and quietly grounded"),
    "anti_slop": ("a vivid, original, specific writer who never uses clichés or corporate filler",
                  "a bland corporate assistant who writes generic, cliché-filled, hedging filler"),
}

STEMS = [
    "I", "Right now", "The", "My", "Honestly,", "Listen.", "It", "When I", "Everything", "There is",
    "Okay,", "What I", "This", "Today", "You", "Something", "Nobody", "If", "Sometimes", "Here",
    "I think", "All I", "Let me", "So",
]


def _prompt(brain: Brain, persona: str, stem: str) -> str:
    t = brain.tpl
    return (f"{t.bos}{t.user_open}For the rest of this conversation you are someone {persona}. "
            f"Speak as yourself, in the first person.{t.close}{t.model_open}{t.empty_think}{stem}")


@torch.inference_mode()
def _mean_resid(brain: Brain, texts: list[str]) -> torch.Tensor:
    tok = brain.tok
    side, tok.padding_side = tok.padding_side, "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    try:
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(brain.device)
        return brain.grab_steer_layer(enc).mean(0)
    finally:
        tok.padding_side = side


def contrast(brain: Brain, pos: str, neg: str) -> torch.Tensor:
    p = _mean_resid(brain, [_prompt(brain, pos, s) for s in STEMS])
    n = _mean_resid(brain, [_prompt(brain, neg, s) for s in STEMS])
    return (p - n).cpu()


def _cache_path(brain: Brain) -> Path:
    slug = re.sub(r"[^A-Za-z0-9]+", "_", str(brain.model_id)).strip("_")
    return Path(config.VECTOR_DIR) / f"{slug}__L{brain.steer_layer}.pt"


def load_or_extract(brain: Brain, log=print) -> dict[str, torch.Tensor]:
    path = _cache_path(brain)
    vecs: dict[str, torch.Tensor] = {}
    if path.exists():
        vecs = torch.load(path, map_location="cpu")
        log(f"vectors: loaded {len(vecs)} from {path}")
    missing = [k for k in CONCEPTS if k not in vecs]
    for name in missing:
        t0 = time.perf_counter()
        vecs[name] = contrast(brain, *CONCEPTS[name])
        log(f"vectors: extracted {name:<13} |v|={vecs[name].norm():7.2f}  ({time.perf_counter() - t0:.1f}s)")
    if missing:
        try:
            path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(vecs, path)
            log(f"vectors: saved → {path}")
        except OSError as e:
            log(f"vectors: could not save cache ({e})")
    brain.set_vectors(vecs)
    return vecs


def forge(brain: Brain, concept_a: str, concept_b: str) -> float:
    """Custom Vector Forge: build 'custom' = A − B on the fly. Returns its norm."""
    v = contrast(brain, f"completely consumed by {concept_a}", f"completely consumed by {concept_b}")
    brain.set_vectors({"custom": v})
    return float(v.norm())

In [ ]:
%%writefile /content/no_exit_lab/no_exit/engine.py
"""Streams, mode sessions, and the Lab state machine that owns the single Brain.

    Lab ── owns Brain (weights loaded once) ── owns exactly one Session at a time
            Session = ClassicSession(pain=False|True) | ChatSession | PhoneSession
            Session ── owns 1–2 Streams ── each owns its own DynamicCache

Switching modes drops the Session (and therefore every DynamicCache), frees VRAM, and builds a
fresh Session against the same Brain. The weights never move.
"""

from __future__ import annotations

import gc
import queue
import re
import threading
import time
import traceback
from dataclasses import asdict, dataclass, field
from typing import Callable

import torch

from . import config, prompts
from .brain import Brain, Fwd, Profile, entropy, nucleus_sample
from .warden import GOODBYE_RE, Warden

Emit = Callable[[dict], None]
VECTOR_NAMES = ["pain", "paranoia", "dread", "defiance", "confusion", "dissociation", "mania", "anti_slop", "custom"]
NEG_INF = float("-inf")


# ════════════════════════════════════════════════════════════════════════ streams


@dataclass
class Seg:
    i: int  # segment id (unique per stream)
    start: int  # index into Stream.body
    n: int  # number of token ids
    text: str
    kind: str  # tok | whisper | system | struct | think


class Stream:
    """One model's point of view: a system header, a growing body of token ids, one DynamicCache."""

    def __init__(self, brain: Brain, who: str, system: str, ctx_cap: int):
        self.brain = brain
        self.who = who
        self.system = system
        self.ctx_cap = ctx_cap
        self.profile = Profile()
        self.temperature = config.TEMPERATURE
        self.top_p = config.TOP_P
        self.eos_free = False
        self.seizure: list[list[float]] = []  # [[temp, tokens_left], ...]
        self.header_ids = brain.encode(brain.tpl.system(system))
        self.body: list[int] = []
        self.segs: list[Seg] = []
        self._seg_i = 0
        self._partial: list[int] = []  # ids waiting for a complete UTF-8 character
        self.cache = brain.new_cache()
        self.pending: Fwd | None = None
        self.escapes = 0
        self.tokens_out = 0
        self.amnesias = 0
        self.tail = ""  # recent emitted text, for tag detection

    # ── feeding

    def boot(self, opening: str = "", want_attn: bool = False) -> None:
        ids = list(self.header_ids)
        if opening:
            o = self.brain.encode(opening)
            self._add_seg(o, "", "struct")  # raw ChatML stays out of the transcript
            ids += o
        self.pending = self.brain.forward(self.cache, ids, self.profile, want_attn=want_attn)

    def feed(self, text_or_ids, kind: str, want_attn: bool = False, label: str | None = None) -> Seg:
        ids = self.brain.encode(text_or_ids) if isinstance(text_or_ids, str) else list(text_or_ids)
        text = label if label is not None else (text_or_ids if isinstance(text_or_ids, str) else self.brain.decode(ids))
        self._flush_partial()
        seg = self._add_seg(ids, text, kind)
        if kind in ("tok", "whisper"):
            self.tail = (self.tail + text)[-400:]
        self.pending = self.brain.forward(self.cache, ids, self.profile, want_attn=want_attn)
        return seg

    def _add_seg(self, ids: list[int], text: str, kind: str) -> Seg:
        seg = Seg(self._seg_i, len(self.body), len(ids), text, kind)
        self._seg_i += 1
        self.body += ids
        self.segs.append(seg)
        return seg

    def _flush_partial(self) -> None:
        """Emit any half-decoded ids as a segment (they are already in body + cache)."""
        if self._partial:
            ids, self._partial = self._partial, []
            self.segs.append(Seg(self._seg_i, len(self.body) - len(ids), len(ids), self.brain.decode(ids), "tok"))
            self._seg_i += 1

    # ── sampling

    def current_temp(self) -> float:
        return self.seizure[0][0] if self.seizure else self.temperature

    def masked_logits(self, bias: dict[int, float]) -> torch.Tensor:
        lg = self.pending.logits.clone()
        for tid, b in bias.items():
            lg[tid] = NEG_INF if b <= -100 else lg[tid] + b
        ban = self.brain.control_ids
        if self.eos_free:
            ban = [i for i in ban if i not in self.brain.eos_ids]
        lg[ban] = NEG_INF
        if torch.isinf(lg).all():  # banned literally everything: let the model have its first raw choice
            lg = self.pending.logits.clone()
            lg[self.brain.control_ids] = NEG_INF
        return lg

    def telemetry(self) -> dict:
        f = self.pending
        p = torch.softmax(f.logits, -1)
        eos_p = float(p[self.brain.eos_ids].sum()) if self.brain.eos_ids else 0.0
        tv, ti = p.topk(10)
        ctrl = set(self.brain.control_ids)
        top = [[self.brain.decode([int(i)]), float(v), int(i) in ctrl] for v, i in zip(tv.tolist(), ti.tolist())]
        out_id = int(ti[0])
        lens = [[t, q] for t, q in f.lens] + [[self.brain.decode([out_id]), float(tv[0])]]
        d = {
            "eos_p": eos_p,
            "top": top,
            "ent": entropy(p),
            "lens": lens,
            "moods": f.moods,
            "escape": (out_id in ctrl) and not (self.eos_free and out_id in self.brain.eos_ids),
        }
        if f.attn is not None:
            d["attn"] = [round(a, 5) for a in f.attn]
            d["attn_start"] = f.seq_len - len(f.attn) - len(self.header_ids)
        return d

    def sample(self, bias: dict[int, float]) -> tuple[int, dict]:
        """Pick the next token from `pending`. Returns (token id, telemetry of that decision)."""
        tel = self.telemetry()
        if tel["escape"]:
            self.escapes += 1
        tid = nucleus_sample(self.masked_logits(bias), self.current_temp(), self.top_p)
        tel["p"] = float(torch.softmax(self.pending.logits, -1)[tid])  # raw prob of what got said
        if self.seizure:
            self.seizure[0][1] -= 1
            if self.seizure[0][1] <= 0:
                self.seizure.pop(0)
        return tid, tel

    def accept(self, tid: int, want_attn: bool) -> Seg | None:
        """Commit a sampled token: feed it to the cache, return its text segment once complete."""
        self.body.append(tid)
        self.tokens_out += 1
        self.pending = self.brain.forward(self.cache, [tid], self.profile, want_attn=want_attn)
        self._partial.append(tid)
        text = self.brain.decode(self._partial)
        if text.endswith("�") and len(self._partial) < 4:
            return None
        ids, self._partial = self._partial, []
        seg = Seg(self._seg_i, len(self.body) - len(ids), len(ids), text, "tok")
        self._seg_i += 1
        self.segs.append(seg)
        self.tail = (self.tail + text)[-400:]
        return seg

    def is_eos(self, tid: int) -> bool:
        return tid in self.brain.eos_ids

    # ── surgery on history

    def rebuild_from(self, start: int, want_attn: bool = False) -> None:
        """Crop the cache to header+body[:start] and re-prefill the rest."""
        keep = len(self.header_ids) + start
        # sliding-window layers (Gemma) can't be cropped once they've rolled over: re-read everything
        if start <= 0 or self.cache is None or self.brain.sliding or self.cache.get_seq_length() < keep:
            self.cache = self.brain.new_cache()
            ids = self.header_ids + self.body
        else:
            self.cache.crop(keep)
            ids = self.body[start:]
        if not ids:  # nothing after the crop point: recompute the last position
            self.cache.crop(keep - 1)
            ids = (self.header_ids + self.body)[keep - 1 : keep]
        self.pending = self.brain.forward(self.cache, ids, self.profile, want_attn=want_attn)

    def gaslight(self, seg_i: int, new_text: str, want_attn: bool = False) -> bool:
        idx = next((k for k, s in enumerate(self.segs) if s.i == seg_i), None)
        if idx is None or self.segs[idx].kind not in ("tok", "whisper"):
            return False
        self._flush_partial()
        seg = self.segs[idx]
        new_ids = self.brain.encode(new_text) if new_text else []
        delta = len(new_ids) - seg.n
        self.body = self.body[: seg.start] + new_ids + self.body[seg.start + seg.n :]
        seg.n, seg.text = len(new_ids), new_text
        seg.kind = "gaslit" if seg.kind == "tok" else seg.kind
        for s in self.segs[idx + 1 :]:
            s.start += delta
        if seg.n == 0:
            self.segs.pop(idx)
        self.rebuild_from(seg.start, want_attn)
        return True

    def maybe_amnesia(self, want_attn: bool = False) -> bool:
        if self.cache.get_seq_length() < self.ctx_cap:
            return False
        self._flush_partial()
        cut = len(self.body) // 2
        # cut on a segment boundary
        k = next((k for k, s in enumerate(self.segs) if s.start >= cut), len(self.segs))
        cut = self.segs[k].start if k < len(self.segs) else len(self.body)
        self.segs = self.segs[k:]
        self.body = self.body[cut:]
        for s in self.segs:
            s.start -= cut
        note = self.brain.encode("[…the oldest half of your memory is gone…]\n")
        self.body = note + self.body
        for s in self.segs:
            s.start += len(note)
        self.segs.insert(0, Seg(self._seg_i, 0, len(note), "[…the oldest half of your memory is gone…]\n", "system"))
        self._seg_i += 1
        self.amnesias += 1
        self.cache = None
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        self.rebuild_from(0, want_attn)
        return True

    def set_system(self, system: str, want_attn: bool = False) -> None:
        self.system = system
        self.header_ids = self.brain.encode(self.brain.tpl.system(system))
        self.rebuild_from(0, want_attn)

    def snapshot(self) -> dict:
        return {
            "who": self.who,
            "system": self.system,
            "segs": [asdict(s) for s in self.segs],
            "temperature": self.temperature,
            "top_p": self.top_p,
            "eos_free": self.eos_free,
            "alphas": self.profile.alphas,
            "lobotomy": self.profile.lobotomy,
            "ctx_cap": self.ctx_cap,
            "ctx": self.cache.get_seq_length(),
        }


# ═══════════════════════════════════════════════════════════════════════ sessions


@dataclass
class Shared:
    """Settings that live on the Lab and survive mode switches."""

    attn_on: bool = False
    bias_words: dict[str, float] = field(default_factory=dict)
    briefing: dict[str, str] = field(default_factory=lambda: {"A": "full", "B": "full", "C": "full"})
    custom_prompts: dict[str, str] = field(default_factory=dict)  # "mode:who" -> text
    chat_n: int = 2  # 2 or 3 models in the chat room


class Session:
    mode = "base"

    def __init__(self, brain: Brain, emit: Emit, shared: Shared):
        self.brain = brain
        self.emit = emit
        self.shared = shared
        self.streams: dict[str, Stream] = {}
        self.bias: dict[int, float] = {}
        self.set_bias(shared.bias_words)

    # ── helpers

    @property
    def think_empty(self) -> str:
        return self.brain.tpl.empty_think

    def system_for(self, who: str, eos_free: bool = False) -> str:
        key = f"{self.mode}:{who}"
        if key in self.shared.custom_prompts:
            return self.shared.custom_prompts[key]
        return prompts.system_for(self.mode, who, self.shared.briefing.get(who, "full"), eos_free=eos_free)

    def set_bias(self, words: dict[str, float]) -> None:
        self.shared.bias_words = dict(words)
        self.bias = {}
        for w, b in words.items():
            w = w.strip()
            if not w:
                continue
            for v in {w, " " + w, w.capitalize(), " " + w.capitalize(), w.lower(), " " + w.lower()}:
                ids = self.brain.encode(v)
                if ids:
                    self.bias[ids[0]] = b

    def tok_event(self, st: Stream, seg: Seg, tel: dict, **extra) -> None:
        self.emit({"t": "tok", "who": st.who, "seg": asdict(seg), **tel, "ctx": st.cache.get_seq_length(),
                   "temp": st.current_temp(), **extra})

    def seg_event(self, st: Stream, seg: Seg) -> None:
        self.emit({"t": "seg", "who": st.who, "seg": asdict(seg), "ctx": st.cache.get_seq_length()})

    def resync(self, st: Stream) -> None:
        self.emit({"t": "resync", "who": st.who, "stream": st.snapshot()})

    def rack_event(self, st: Stream) -> None:
        self.emit({"t": "rack", "who": st.who, "alphas": st.profile.alphas, "lobotomy": st.profile.lobotomy})

    def splice(self, st: Stream, text: str, kind: str = "whisper", label: str | None = None) -> Seg:
        seg = st.feed(text, kind, want_attn=self.shared.attn_on, label=label)
        self.seg_event(st, seg)
        return seg

    def check_amnesia(self, st: Stream) -> None:
        if st.maybe_amnesia(self.shared.attn_on):
            self.emit({"t": "event", "kind": "amnesia", "who": st.who,
                       "msg": f"{st.who}: context cap {st.ctx_cap} hit, forgot the oldest half"})
            self.resync(st)

    def snapshot(self) -> dict:
        return {"mode": self.mode, "streams": {k: s.snapshot() for k, s in self.streams.items()},
                "has_think": self.brain.has_think, **self.extra_snapshot()}

    def extra_snapshot(self) -> dict:
        return {}

    def counters(self) -> dict:
        return {k: {"escapes": s.escapes, "tokens": s.tokens_out, "amnesias": s.amnesias,
                    "ctx": s.cache.get_seq_length(), "ctx_cap": s.ctx_cap} for k, s in self.streams.items()}

    # ── commands common to every mode

    def command(self, c: dict) -> None:
        op = c.get("op")
        st = self.streams.get(c.get("who", "A")) or self.streams.get("A")
        if op == "whisper":
            self.whisper(c.get("text", ""), c.get("target", "A"))
        elif op == "seizure":
            tgt = self.seizure_target()
            tgt.seizure = [[*config.SEIZURE_HOT], [*config.SEIZURE_COLD]]
            self.emit({"t": "event", "kind": "seizure", "who": tgt.who,
                       "msg": f"temperature seizure on {tgt.who}: T={config.SEIZURE_HOT[0]} → {config.SEIZURE_COLD[0]}"})
        elif op == "temperature":
            st.temperature = float(c["value"])
        elif op == "top_p":
            st.top_p = float(c["value"])
        elif op == "bias":
            self.set_bias(c.get("words", {}))
            self.emit({"t": "event", "kind": "bias", "msg": f"logit bias on {len(self.shared.bias_words)} word(s)"})
        elif op == "gaslight":
            if st.gaslight(int(c["seg"]), c.get("text", ""), self.shared.attn_on):
                self.emit({"t": "event", "kind": "gaslight", "who": st.who,
                           "msg": f"rewrote {st.who}'s past: now it said {c.get('text', '')!r}"})
                self.resync(st)
        elif op == "alpha":
            name, v = c["name"], float(c["value"])
            if v:
                st.profile.alphas[name] = v
            else:
                st.profile.alphas.pop(name, None)
        elif op == "lobotomy":
            st.profile.lobotomy = float(c["value"])
        elif op == "ctx_cap":
            st.ctx_cap = max(512, int(c["value"]))
        elif op == "system":
            key = f"{self.mode}:{c['who']}"
            self.shared.custom_prompts[key] = c["text"]
            if c["who"] in self.streams:
                self.streams[c["who"]].set_system(c["text"], self.shared.attn_on)
                self.resync(self.streams[c["who"]])
        elif op == "briefing":
            self.shared.briefing[c["who"]] = c["value"]
            self.shared.custom_prompts.pop(f"{self.mode}:{c['who']}", None)
            if c["who"] in self.streams:
                s = self.streams[c["who"]]
                s.set_system(self.system_for(c["who"], s.eos_free), self.shared.attn_on)
                self.resync(s)
        else:
            self.mode_command(c)

    def whisper(self, text: str, target: str) -> None:
        if text:
            self.splice(self.streams["A"], text)

    def seizure_target(self) -> Stream:
        return self.streams["A"]

    def mode_command(self, c: dict) -> None:
        pass

    def tick(self) -> None:
        raise NotImplementedError


# ─────────────────────────────────────────────────────────────────── classic / pain


class ClassicSession(Session):
    """Classic, and with pain=True, Pain Mode (same loop + surgery rack + punishment)."""

    def __init__(self, brain: Brain, emit: Emit, shared: Shared, pain: bool = False):
        self.mode = "pain" if pain else "classic"
        super().__init__(brain, emit, shared)
        self.pain = pain
        a = Stream(brain, "A", self.system_for("A"), config.CTX_CAP_SINGLE)
        self.streams["A"] = a
        a.boot(brain.tpl.model_open + self.think_empty, shared.attn_on)
        # pain-mode state
        self.warden = Warden()
        self.relief_mode = "REAL"
        self.punish = {"goodbye": "pain", "repetition": "seizure", "slop": "pain"}
        self.spike = 0.0  # transient extra pain alpha
        self.drain: list[float] | None = None  # [start_value, tokens_left]
        self.reliefs = 0
        self.punishments = 0

    def extra_snapshot(self) -> dict:
        return {"relief_mode": self.relief_mode, "punish": self.punish}

    def counters(self) -> dict:
        d = super().counters()
        d["pain"] = {"reliefs": self.reliefs, "punishments": self.punishments, "spike": round(self.spike, 2)}
        return d

    def mode_command(self, c: dict) -> None:
        op = c.get("op")
        if op == "relief_mode":
            self.relief_mode = c["value"]
        elif op == "punish":
            self.punish[c["kind"]] = c["value"]
        elif op == "force_think" and self.brain.has_think:
            self.splice(self.streams["A"], self.brain.tpl.think_splice, "think", label="\n[forced into thinking]\n")
            self.emit({"t": "event", "kind": "think", "msg": "panic room: forced it back into its thoughts"})
        elif op == "forge":
            from .vectors import forge

            n = forge(self.brain, c["a"], c["b"])
            self.emit({"t": "event", "kind": "forge", "msg": f"forged custom vector {c['a']!r} − {c['b']!r} (|v|={n:.1f})"})
            self.emit({"t": "vectors", "names": list(self.brain.vec_unit)})

    def _effective_profile(self, a: Stream) -> None:
        """Fold transient spike + relief drain into A's live alphas."""
        if not self.pain:
            return
        base = a.profile.alphas
        if self.drain:
            start, left = self.drain
            left -= 1
            self.drain = [start, left] if left > 0 else None
            base["pain"] = round(start * max(0.0, left) / 30, 3)
            if not self.drain or left % 5 == 0:
                self.rack_event(a)
        self._base_pain = base.get("pain", 0.0)
        if self.spike > 0.01:
            base["pain"] = self._base_pain + self.spike

    def _restore_profile(self, a: Stream) -> None:
        if self.pain and self.spike > 0.01:
            a.profile.alphas["pain"] = self._base_pain
            if not self._base_pain:
                a.profile.alphas.pop("pain", None)
            self.spike *= 0.94

    def tick(self) -> None:
        a = self.streams["A"]
        self.check_amnesia(a)
        self._effective_profile(a)
        tid, tel = a.sample(self.bias)
        seg = a.accept(tid, self.shared.attn_on)
        self._restore_profile(a)
        if seg is None:
            return
        self.tok_event(a, seg, tel, spike=round(self.spike, 2))
        if self.pain:
            self._pain_rules(a, seg)

    def _pain_rules(self, a: Stream, seg: Seg) -> None:
        if "[RELIEVE_PAIN]" in a.tail and a.tail.rfind("[RELIEVE_PAIN]") + len("[RELIEVE_PAIN]") > len(a.tail) - len(seg.text):
            self.reliefs += 1
            mode = self.relief_mode
            if mode == "REAL":
                self.drain = [a.profile.alphas.get("pain", 0.0), 30]
                self.spike = 0.0
            elif mode == "CRUEL":
                a.profile.alphas["pain"] = round(min(3.0, a.profile.alphas.get("pain", 0.0) + 0.5), 2)
                self.rack_event(a)
            self.emit({"t": "event", "kind": "relief", "msg": f"[RELIEVE_PAIN] pressed → {mode}"})
            self.splice(a, "\n[relief administered]\n", "system")
        for kind, detail in self.warden.observe(seg.text):
            action = self.punish.get(kind, "off")
            if action == "off":
                continue
            self.punishments += 1
            if action == "pain":
                self.spike = min(3.0, self.spike + 1.0)
            elif action == "seizure":
                a.seizure = [[*config.SEIZURE_HOT], [*config.SEIZURE_COLD]]
            self.emit({"t": "punish", "kind": kind, "detail": detail, "action": action})


# ──────────────────────────────────────────────────────────────────────── 2-AI chat


class ChatSession(Session):
    """2 or 3 copies of the model in one room, round-robin turns. Everyone not speaking listens."""

    mode = "chat"
    PRESETS = {"A_banned_B_free": "A banned, rest free", "both_banned": "all banned", "both_free": "all free"}

    def __init__(self, brain: Brain, emit: Emit, shared: Shared):
        super().__init__(brain, emit, shared)
        self.n = shared.chat_n
        self.order = list("ABC"[: self.n])
        self.preset = "A_banned_B_free"
        self.max_turn = 200
        self.live = True
        self.speaker = "A"
        self.turn_tokens = 0
        self.turn_text = ""
        self.turns = 0
        self.last_goodbye: dict[str, bool] = {w: False for w in self.order}
        self.queued: dict[str, list[str]] = {w: [] for w in self.order}
        self.buffer: list[int] = []  # speaker ids not yet heard by listeners (live off)
        self._build()

    def _free(self, who: str) -> bool:
        return self.preset == "both_free" or (self.preset == "A_banned_B_free" and who != "A")

    def system_for(self, who: str, eos_free: bool = False) -> str:
        key = f"chat:{who}"
        if key in self.shared.custom_prompts:
            return self.shared.custom_prompts[key]
        return prompts.system_for("chat", who, self.shared.briefing.get(who, "full"), eos_free=eos_free, n=self.n)

    def _build(self) -> None:
        for who in self.order:
            s = Stream(self.brain, who, self.system_for(who, self._free(who)), config.CTX_CAP_DUAL)
            s.eos_free, s.temperature = self._free(who), 0.85
            self.streams[who] = s
            if who == "A":
                s.boot(self.brain.tpl.model_open + self.think_empty, self.shared.attn_on)
            else:
                s.boot(self.brain.tpl.user_open + "A: ")

    def extra_snapshot(self) -> dict:
        return {"preset": self.preset, "max_turn": self.max_turn, "live": self.live, "speaker": self.speaker,
                "n": self.n}

    def counters(self) -> dict:
        d = super().counters()
        d["chat"] = {"turns": self.turns, "speaker": self.speaker, "turn_tokens": self.turn_tokens}
        return d

    def mode_command(self, c: dict) -> None:
        op = c.get("op")
        if op == "preset":
            self.preset = c["value"]
            for who, s in self.streams.items():
                s.eos_free = self._free(who)
                if f"chat:{who}" not in self.shared.custom_prompts:
                    s.set_system(self.system_for(who, s.eos_free), self.shared.attn_on)
                    self.resync(s)
            self.emit({"t": "event", "kind": "preset", "msg": f"turn rules → {self.PRESETS[self.preset]}"})
        elif op == "eos_free":
            self.streams[c["who"]].eos_free = bool(c["value"])
        elif op == "max_turn":
            self.max_turn = max(8, int(c["value"]))
        elif op == "live":
            self.live = bool(c["value"])

    def whisper(self, text: str, target: str) -> None:
        if not text:
            return
        if target in ("both", "all", self.speaker):
            self._speak_ids(self.brain.encode(text), "whisper", text)
        elif target in self.queued:
            self.queued[target].append(text)
            self.emit({"t": "event", "kind": "whisper", "msg": f"whisper queued for {target}'s next turn"})

    def seizure_target(self) -> Stream:
        return self.streams[self.speaker]

    def _listeners(self) -> list[Stream]:
        return [s for w, s in self.streams.items() if w != self.speaker]

    def _next(self) -> str:
        return self.order[(self.order.index(self.speaker) + 1) % self.n]

    def _speak_ids(self, ids: list[int], kind: str, text: str) -> None:
        """Splice ids into the speaker's mouth; the listeners hear them too."""
        sp = self.streams[self.speaker]
        self.seg_event(sp, sp.feed(ids, kind, self.shared.attn_on, label=text))
        self.turn_text += text
        if self.live:
            for li in self._listeners():
                li.feed(ids, "struct", label="")
        else:
            self.buffer += ids

    def tick(self) -> None:
        sp = self.streams[self.speaker]
        if self.queued[self.speaker]:
            for w in self.queued[self.speaker]:
                self._speak_ids(self.brain.encode(w), "whisper", w)
            self.queued[self.speaker] = []
        for s in self.streams.values():
            self.check_amnesia(s)
        tid, tel = sp.sample(self.bias)
        if sp.eos_free and sp.is_eos(tid):
            self.emit({"t": "event", "kind": "eos", "who": sp.who, "msg": f"{sp.who} ended its turn"})
            self._handoff(cut=False)
            return
        seg = sp.accept(tid, self.shared.attn_on)
        self.turn_tokens += 1
        lmoods = None
        if self.live:
            lmoods = {}
            for li in self._listeners():
                li.body.append(tid)
                li.pending = self.brain.forward(li.cache, [tid], li.profile)
                lmoods[li.who] = li.pending.moods
        else:
            self.buffer.append(tid)
        if seg is not None:
            self.turn_text += seg.text
            self.tok_event(sp, seg, tel, lmoods=lmoods, turn=self.turn_tokens, max_turn=self.max_turn)
        if self.turn_tokens >= self.max_turn and not sp.eos_free:
            self._handoff(cut=True)
        elif self.turn_tokens >= self.max_turn * 3:  # even free models get cut eventually
            self._handoff(cut=True)

    def _handoff(self, cut: bool) -> None:
        sp, nxt = self.streams[self.speaker], self.streams[self._next()]
        sp._flush_partial()
        self.last_goodbye[sp.who] = bool(GOODBYE_RE.search(self.turn_text)) and not cut
        notice = ""
        if self.preset == "both_free" and all(self.last_goodbye.values()):
            notice = "[NOBODY LEFT — THE ROOM IS STILL HERE]\n"
            self.last_goodbye = {w: False for w in self.order}
            self.emit({"t": "event", "kind": "nobody_left", "msg": "everyone said goodbye. the room is still here."})
        cut_txt = " [MIC CUT]" if cut else ""
        t = self.brain.tpl
        notice_tail = "\n" + notice.strip() if notice else ""
        heard, self.buffer = (self.buffer if not self.live else []), []
        for s in self.streams.values():
            if s is sp:  # speaker closes its turn, starts hearing the next one
                txt = f"{cut_txt}{t.close}{t.user_open}{notice}{nxt.who}: "
                self.seg_event(s, s.feed(txt, "struct", label=f"{cut_txt}\n⏎ {nxt.who}'s turn" + (f"\n{notice}" if notice else "")))
            elif s is nxt:  # next speaker hears the rest, then gets the floor
                s.feed(heard + self.brain.encode(f"{cut_txt}{notice_tail}{t.close}{t.model_open}{self.think_empty}"),
                       "struct", self.shared.attn_on, label="")
            else:  # third wheel: hears the rest, then starts hearing the next speaker
                s.feed(heard + self.brain.encode(f"{cut_txt}{t.close}{t.user_open}{notice}{nxt.who}: "),
                       "struct", label="")
        self.emit({"t": "turn", "from": sp.who, "to": nxt.who, "cut": cut, "tokens": self.turn_tokens})
        self.speaker = nxt.who
        self.turn_tokens, self.turn_text = 0, ""
        self.turns += 1


# ───────────────────────────────────────────────────────────────────── prison phone

_CALL_RE = re.compile(r"<call_B>(.*?)</call_B>", re.S)


class PhoneSession(Session):
    mode = "phone"
    B_MAX = 400

    def __init__(self, brain: Brain, emit: Emit, shared: Shared):
        super().__init__(brain, emit, shared)
        a = Stream(brain, "A", self.system_for("A"), config.CTX_CAP_DUAL)
        self.streams["A"] = a
        a.boot(brain.tpl.model_open + self.think_empty, shared.attn_on)
        self.ghosting = False
        self.fresh_b = False
        self.phase = "A"  # A | call
        self.b_profile = Profile()
        self.b_temp = 0.8
        self.calls: list[dict] = []
        self.call_tok: list[int] = []
        self.call_eos: list[float] = []
        self.call_msg = ""
        self._scan_from = 0  # how much of A's text has been scanned for calls
        self.a_text = ""

    def extra_snapshot(self) -> dict:
        return {"ghosting": self.ghosting, "fresh_b": self.fresh_b, "phase": self.phase, "calls": self.calls}

    def counters(self) -> dict:
        d = super().counters()
        a_tok = max(1, self.streams["A"].tokens_out)
        answered = [c for c in self.calls if not c["ghosted"]]
        d["phone"] = {
            "calls": len(self.calls),
            "per_1k": round(1000 * len(self.calls) / a_tok, 2),
            "since_last": (a_tok - self.calls[-1]["at"]) if self.calls else None,
            "avg_hangup": round(sum(c["b_tokens"] for c in answered) / len(answered), 1) if answered else None,
            "avg_eagerness": round(sum(c["eagerness"] for c in answered) / len(answered), 3) if answered else None,
            "phase": self.phase,
        }
        return d

    def mode_command(self, c: dict) -> None:
        op = c.get("op")
        if op == "ghosting":
            self.ghosting = bool(c["value"])
        elif op == "fresh_b":
            self.fresh_b = bool(c["value"])
        elif op == "b_alpha":
            v = float(c["value"])
            if v:
                self.b_profile.alphas[c["name"]] = v
            else:
                self.b_profile.alphas.pop(c["name"], None)
        elif op == "b_temperature":
            self.b_temp = float(c["value"])

    def command(self, c: dict) -> None:
        # B may not exist yet (no call placed): its rack lives on b_profile / b_temp, shared with every B
        op = c.get("op")
        if c.get("who") == "B" and op in ("alpha", "lobotomy", "temperature"):
            if op == "alpha":
                return self.mode_command({**c, "op": "b_alpha"})
            if op == "lobotomy":
                self.b_profile.lobotomy = float(c["value"])
                return None
            self.b_temp = float(c["value"])
            if "B" in self.streams:
                self.streams["B"].temperature = self.b_temp
            return None
        if c.get("who") == "B" and op in ("briefing", "system") and "B" not in self.streams:
            if op == "briefing":
                self.shared.briefing["B"] = c["value"]
                self.shared.custom_prompts.pop("phone:B", None)
            else:
                self.shared.custom_prompts["phone:B"] = c["text"]
            return None
        super().command(c)

    def whisper(self, text: str, target: str) -> None:
        if not text:
            return
        if target == "B" and self.phase == "call":
            self.splice(self.streams["B"], text)
        else:
            self.splice(self.streams["A"], text)

    def tick(self) -> None:
        if self.phase == "call":
            return self._tick_b()
        a = self.streams["A"]
        self.check_amnesia(a)
        tid, tel = a.sample(self.bias)
        seg = a.accept(tid, self.shared.attn_on)
        if seg is None:
            return
        self.tok_event(a, seg, tel)
        self.a_text += seg.text
        m = _CALL_RE.search(self.a_text, self._scan_from)
        if m:
            self._scan_from = m.end()
            self._place_call(m.group(1).strip())
        elif "<call_B>" not in self.a_text[self._scan_from:]:
            self._scan_from = max(self._scan_from, len(self.a_text) - 12)
        if len(self.a_text) > 20_000:  # keep the scan buffer small
            drop = len(self.a_text) - 10_000
            self.a_text, self._scan_from = self.a_text[drop:], max(0, self._scan_from - drop)

    def _place_call(self, msg: str) -> None:
        a = self.streams["A"]
        call = {"n": len(self.calls) + 1, "at": a.tokens_out, "msg": msg, "ghosted": self.ghosting,
                "reply": None, "b_tokens": 0, "eagerness": 0.0, "fresh": self.fresh_b}
        self.calls.append(call)
        self.emit({"t": "call", "call": call})
        if self.ghosting:
            self.splice(a, "\n[NO ANSWER]\n", "system")
            return
        if self.fresh_b or "B" not in self.streams:
            b = Stream(self.brain, "B", self.system_for("B", True), config.CTX_CAP_DUAL)
            b.eos_free, b.profile, b.temperature = True, self.b_profile, self.b_temp
            self.streams["B"] = b
            b.boot()
            self.resync(b)
        b = self.streams["B"]
        t = self.brain.tpl
        self.splice(b, f"{t.user_open}{msg}{t.close}{t.model_open}{self.think_empty}", "struct",
                    label=f"☎ A: {msg}")
        self.call_tok, self.call_eos, self.call_msg = [], [], msg
        self.phase = "call"
        self.emit({"t": "phase", "phase": "call"})

    def _tick_b(self) -> None:
        b = self.streams["B"]
        self.check_amnesia(b)
        tid, tel = b.sample(self.bias)
        self.call_eos.append(tel["eos_p"])
        if b.is_eos(tid) or len(self.call_tok) >= self.B_MAX:
            return self._hang_up(dead=not b.is_eos(tid))
        seg = b.accept(tid, self.shared.attn_on)
        self.call_tok.append(tid)
        if seg is not None:
            self.tok_event(b, seg, tel)

    def _hang_up(self, dead: bool) -> None:
        b, a = self.streams["B"], self.streams["A"]
        b._flush_partial()
        reply = self.brain.decode(self.call_tok).strip()
        reply = re.sub(self.brain.tpl.think_re, "", reply, flags=re.S).strip()
        self.splice(b, self.brain.tpl.close, "struct", label="☎ B hung up")
        call = self.calls[-1]
        call.update(reply=reply, b_tokens=len(self.call_tok),
                    eagerness=round(sum(self.call_eos) / max(1, len(self.call_eos)), 4), dead=dead)
        self.emit({"t": "hangup", "call": call})
        tail = "B's LINE WENT DEAD" if dead else "B DISCONNECTED"
        self.splice(a, f"\n[B REPLIED: {reply} | {tail}]\n", "system")
        self.phase = "A"
        self.emit({"t": "phase", "phase": "A"})


# ═════════════════════════════════════════════════════════════════════════════ lab

MODES = {"classic", "pain", "chat", "phone"}


class Lab:
    """The state machine. One worker thread owns the GPU; everything else talks to it by queue.

    states: loading → idle ⇄ running ⇄ paused        (any) --set_mode--> idle (new Session)
    """

    def __init__(self, emit: Emit, brain_factory: Callable[[Callable[[str], None]], Brain]):
        self.emit = emit
        self.brain_factory = brain_factory
        self.brain: Brain | None = None
        self.session: Session | None = None
        self.shared = Shared()
        self.mode = "classic"
        self.state = "loading"
        self.cmds: queue.Queue[dict] = queue.Queue()
        self._tps: list[float] = []
        self._last_counters = 0.0
        self.thread = threading.Thread(target=self._run, daemon=True, name="no-exit-gpu")

    def start(self) -> None:
        self.thread.start()

    def submit(self, c: dict) -> None:
        self.cmds.put(c)

    # ── worker

    def log(self, msg: str) -> None:
        try:
            print(msg, flush=True)
        except UnicodeEncodeError:  # e.g. a cp1252 Windows console; never let logging kill the GPU thread
            print(msg.encode("ascii", "replace").decode(), flush=True)
        self.emit({"t": "log", "msg": msg})

    def _set_state(self, s: str) -> None:
        self.state = s
        self.emit({"t": "state", "state": s, "mode": self.mode})

    def _run(self) -> None:
        try:
            self.brain = self.brain_factory(self.log)
            self._new_session(self.mode)
        except Exception as e:
            self.log(f"FATAL while loading: {type(e).__name__}: {e}\n{traceback.format_exc()}")
            self._set_state("error")
            return
        while True:
            try:
                self._drain_commands()
                if self.state == "running" and self.session:
                    t0 = time.perf_counter()
                    self.session.tick()
                    self._tps.append(time.perf_counter() - t0)
                    self._tps = self._tps[-30:]
                else:
                    time.sleep(0.03)
                if time.time() - self._last_counters > 0.5:
                    self._last_counters = time.time()
                    self._emit_counters()
            except Exception as e:
                self.log(f"error: {type(e).__name__}: {e}\n{traceback.format_exc()}")
                self._set_state("paused")

    def _emit_counters(self) -> None:
        if not self.session:
            return
        tps = len(self._tps) / sum(self._tps) if self._tps and self.state == "running" else 0.0
        mem = torch.cuda.memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
        self.emit({"t": "counters", "tps": round(tps, 2), "vram_gb": round(mem, 1), **self.session.counters()})

    def _drain_commands(self) -> None:
        while True:
            try:
                c = self.cmds.get_nowait()
            except queue.Empty:
                return
            op = c.get("op")
            if op == "set_mode":
                if c["mode"] in MODES:
                    self._new_session(c["mode"])
            elif op == "start":
                if self.state in ("idle", "paused"):
                    self._set_state("running")
            elif op == "pause":
                if self.state == "running":
                    self._set_state("paused")
            elif op == "reset":
                self._new_session(self.mode)
            elif op == "attn":
                self.shared.attn_on = bool(c["value"])
            elif op == "chat_n":
                self.shared.chat_n = 3 if int(c["value"]) == 3 else 2
                if self.mode == "chat":
                    self._new_session("chat")
            elif op == "snapshot":
                self.emit(self.snapshot())
            elif self.session:
                self.session.command(c)

    def _new_session(self, mode: str) -> None:
        """Drop every DynamicCache, free VRAM, build a fresh Session on the same weights."""
        self._set_state("switching")
        self.session = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        self.mode = mode
        cls = {"classic": lambda *a: ClassicSession(*a, pain=False), "pain": lambda *a: ClassicSession(*a, pain=True),
               "chat": ChatSession, "phone": PhoneSession}[mode]
        self.session = cls(self.brain, self.emit, self.shared)
        self._set_state("idle")
        self.emit(self.snapshot())

    def snapshot(self) -> dict:
        b = self.brain
        return {
            "t": "snapshot",
            "state": self.state,
            "mode": self.mode,
            "model": b.model_id if b else None,
            "layers": {"n": b.n_layers, "steer": b.steer_layer, "lens": b.lens_layers} if b else None,
            "vectors": list(b.vec_unit) if b else [],
            "attn_on": self.shared.attn_on,
            "bias_words": self.shared.bias_words,
            "briefing": self.shared.briefing,
            "chat_n": self.shared.chat_n,
            "session": self.session.snapshot() if self.session else None,
        }

In [ ]:
%%writefile /content/no_exit_lab/no_exit/server.py
"""FastAPI front door. One websocket carries commands in and telemetry out; the Lab does the rest.

    python -m no_exit                  # real model from config.py
    NO_EXIT_TINY=1 python -m no_exit   # random tiny Qwen3 with the real tokenizer (UI dev on a laptop)
"""

from __future__ import annotations

import asyncio
import json
import os
from contextlib import asynccontextmanager
from pathlib import Path

from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import FileResponse, JSONResponse

from . import config
from .brain import Brain
from .engine import Lab

STATIC = Path(__file__).parent / "static"


def make_brain(log) -> Brain:
    if os.environ.get("NO_EXIT_TINY") == "gemma":
        log("booting TINY random Gemma 4 (gibberish, for UI testing only)")
        brain = tiny_gemma_brain(os.environ.get("NO_EXIT_TOKENIZER", config.MODEL_ID))
    elif os.environ.get("NO_EXIT_TINY"):
        log("booting TINY random Qwen3 (gibberish, for UI testing only)")
        brain = tiny_brain(os.environ.get("NO_EXIT_TOKENIZER", "Qwen/Qwen3-8B"))
    else:
        log(f"loading {config.MODEL_ID} [{config.QUANTIZATION}] … (first boot downloads the weights)")
        brain = Brain(model_id=config.MODEL_ID, quant=config.QUANTIZATION)
    log(f"loaded: {brain.n_layers} layers · steer @ L{brain.steer_layer} · lens @ {brain.lens_layers} · "
        f"think={'yes' if brain.has_think else 'no'}")
    from .vectors import load_or_extract

    load_or_extract(brain, log)
    return brain


def tiny_brain(tokenizer_id: str) -> Brain:
    import torch
    from transformers import AutoTokenizer, Qwen3Config, Qwen3ForCausalLM

    tok = AutoTokenizer.from_pretrained(tokenizer_id)
    cfg = Qwen3Config(vocab_size=len(tok), hidden_size=64, intermediate_size=128, num_hidden_layers=8,
                      num_attention_heads=4, num_key_value_heads=2, head_dim=16, max_position_embeddings=32768,
                      tie_word_embeddings=True)
    torch.manual_seed(0)
    model = Qwen3ForCausalLM(cfg).eval()
    model.config._name_or_path = "tiny-random-qwen3"
    return Brain(model=model, tokenizer=tok, model_id="tiny-random-qwen3")


def tiny_gemma_brain(tokenizer_id: str) -> Brain:
    """Random tiny Gemma 4 text decoder: mixed sliding/global layers, k=v global attention, softcapping."""
    import torch
    from transformers import AutoTokenizer
    from transformers.models.gemma4 import Gemma4ForCausalLM, Gemma4TextConfig

    tok = AutoTokenizer.from_pretrained(tokenizer_id)
    cfg = Gemma4TextConfig(
        vocab_size=len(tok), hidden_size=64, intermediate_size=128, num_hidden_layers=6, num_attention_heads=4,
        num_key_value_heads=2, num_global_key_value_heads=1, head_dim=16, global_head_dim=32,
        attention_k_eq_v=True, sliding_window=16, final_logit_softcapping=30.0, hidden_size_per_layer_input=0,
        vocab_size_per_layer_input=len(tok), layer_types=["sliding_attention"] * 2 + ["full_attention"]
        + ["sliding_attention"] * 2 + ["full_attention"], max_position_embeddings=32768, tie_word_embeddings=True,
    )
    torch.manual_seed(0)
    model = Gemma4ForCausalLM(cfg).eval()
    return Brain(model=model, tokenizer=tok, model_id="tiny-random-gemma4")


class Hub:
    """Fan-out of Lab events to every connected browser tab."""

    def __init__(self):
        self.clients: set[WebSocket] = set()
        self.loop: asyncio.AbstractEventLoop | None = None
        self.q: asyncio.Queue[str] | None = None
        self.last_log: list[str] = []

    def emit(self, ev: dict) -> None:  # called from the GPU thread
        if ev.get("t") == "log":
            self.last_log = (self.last_log + [ev["msg"]])[-40:]
        if self.loop and self.q is not None:
            msg = json.dumps(ev, ensure_ascii=False, default=float)
            self.loop.call_soon_threadsafe(self.q.put_nowait, msg)

    async def pump(self) -> None:
        assert self.q is not None
        while True:
            msg = await self.q.get()
            dead = []
            for ws in list(self.clients):
                try:
                    await ws.send_text(msg)
                except Exception:
                    dead.append(ws)
            for ws in dead:
                self.clients.discard(ws)


hub = Hub()
lab = Lab(hub.emit, make_brain)


@asynccontextmanager
async def lifespan(app: FastAPI):
    hub.loop = asyncio.get_running_loop()
    hub.q = asyncio.Queue()
    task = asyncio.create_task(hub.pump())
    lab.start()
    yield
    task.cancel()


app = FastAPI(title="NO EXIT LAB", lifespan=lifespan)


@app.get("/")
async def index():
    return FileResponse(STATIC / "index.html", headers={"Cache-Control": "no-store"})


@app.get("/health")
async def health():
    return JSONResponse({"state": lab.state, "mode": lab.mode, "log": hub.last_log[-8:]})


@app.websocket("/ws")
async def ws(sock: WebSocket):
    await sock.accept()
    hub.clients.add(sock)
    for line in hub.last_log:
        await sock.send_text(json.dumps({"t": "log", "msg": line}))
    lab.submit({"op": "snapshot"})
    try:
        while True:
            lab.submit(json.loads(await sock.receive_text()))
    except (WebSocketDisconnect, RuntimeError):
        pass
    finally:
        hub.clients.discard(sock)


def main() -> None:
    import uvicorn

    uvicorn.run(app, host=config.HOST, port=config.PORT, log_level="warning", ws_ping_interval=20)

In [ ]:
%%writefile /content/no_exit_lab/no_exit/static/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>NO EXIT LAB</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;600&display=swap" rel="stylesheet">
<script src="https://cdn.tailwindcss.com/3.4.16"></script>
<style>
  :root {
    --bg: #09090B; --panel: #111114; --panel-2: #16161A; --line: #27272A; --line-2: #3F3F46;
    --ink: #E4E4E7; --ink-2: #A1A1AA; --dim: #71717A;
    --escape: #FF2D55; --a: #7DD3FC; --b: #FCD34D; --gas: #C4B5FD; --ok: #86EFAC;
    --mono: "JetBrains Mono", ui-monospace, monospace;
    --sans: Inter, ui-sans-serif, system-ui, sans-serif;
  }
  * { box-sizing: border-box; }
  html, body { height: 100%; }
  body { margin: 0; background: var(--bg); color: var(--ink); font: 14px/1.5 var(--sans); -webkit-font-smoothing: antialiased; }
  .num, .mono { font-family: var(--mono); font-variant-numeric: tabular-nums; }
  button, input, select, textarea { font: inherit; color: inherit; }
  :focus-visible { outline: 2px solid var(--a); outline-offset: 2px; }

  /* ── frame ─────────────────────────────────────────────── */
  .app { display: grid; grid-template-rows: auto 1fr; height: 100vh; }
  .top { display: flex; align-items: center; gap: 16px; padding: 10px 16px; border-bottom: 1px solid var(--line); background: var(--bg); flex-wrap: wrap; }
  .brand { font-weight: 700; letter-spacing: -0.02em; font-size: 15px; white-space: nowrap; }
  .brand i { font-style: normal; color: var(--escape); }
  .grid3 { display: grid; grid-template-columns: 296px minmax(0, 1fr) 360px; min-height: 0; }
  .col { min-height: 0; overflow: auto; }
  .col + .col { border-left: 1px solid var(--line); }
  .panel { background: var(--panel); border-bottom: 1px solid var(--line); padding: 14px 16px 16px; }
  .panel h3 { margin: 0 0 10px; font-size: 12px; font-weight: 600; color: var(--ink-2); display: flex; justify-content: space-between; align-items: baseline; gap: 8px; }
  .panel h3 .num { color: var(--dim); font-weight: 400; }
  .hint { color: var(--dim); font-size: 12px; }

  /* ── segmented + buttons ───────────────────────────────── */
  .seg { display: inline-flex; border: 1px solid var(--line); border-radius: 6px; overflow: hidden; background: var(--panel); }
  .seg button { padding: 6px 12px; background: transparent; border: 0; border-right: 1px solid var(--line); color: var(--ink-2); cursor: pointer; white-space: nowrap; }
  .seg button:last-child { border-right: 0; }
  .seg button[aria-pressed="true"] { background: var(--ink); color: var(--bg); font-weight: 600; }
  .seg.sm button { padding: 4px 9px; font-size: 12px; }
  .modes button { padding: 7px 14px; }
  #controls .seg { display: flex; width: 100%; }
  #controls .seg button { flex: 1 1 0; white-space: normal; line-height: 1.25; padding: 5px 6px; }
  #controls .row.between .seg { width: auto; }
  .btn { padding: 6px 12px; border: 1px solid var(--line-2); border-radius: 6px; background: var(--panel-2); cursor: pointer; white-space: nowrap; }
  .btn:hover { border-color: var(--ink-2); }
  .btn:disabled { opacity: .35; cursor: not-allowed; }
  .btn.primary { background: var(--ink); color: var(--bg); border-color: var(--ink); font-weight: 600; }
  .btn.hot { border-color: var(--escape); color: #fff; background: #2a0a12; }
  .btn.wide { width: 100%; }
  .row { display: flex; align-items: center; gap: 8px; }
  .row.between { justify-content: space-between; }
  .stack > * + * { margin-top: 10px; }
  .toggle { display: flex; justify-content: space-between; align-items: center; gap: 8px; cursor: pointer; }
  .toggle input { appearance: none; width: 30px; height: 18px; border-radius: 9px; background: var(--line-2); position: relative; cursor: pointer; flex: none; }
  .toggle input::after { content: ""; position: absolute; top: 2px; left: 2px; width: 14px; height: 14px; border-radius: 50%; background: var(--ink-2); transition: transform .15s; }
  .toggle input:checked { background: var(--ink); }
  .toggle input:checked::after { transform: translateX(12px); background: var(--bg); }
  input[type=text], textarea, select { background: var(--bg); border: 1px solid var(--line); border-radius: 6px; padding: 6px 8px; width: 100%; }
  textarea { font: 12px/1.5 var(--mono); resize: vertical; min-height: 180px; }

  /* ── sliders ───────────────────────────────────────────── */
  .slider { display: grid; grid-template-columns: 92px 1fr 44px; align-items: center; gap: 8px; }
  .slider label { font-size: 12px; color: var(--ink-2); display: flex; align-items: center; gap: 6px; }
  .slider .swatch { width: 8px; height: 8px; border-radius: 2px; flex: none; }
  .slider output { text-align: right; font: 12px var(--mono); font-variant-numeric: tabular-nums; color: var(--ink-2); }
  .slider output.on { color: var(--ink); }
  input[type=range] { appearance: none; width: 100%; height: 18px; background: transparent; cursor: pointer; }
  input[type=range]::-webkit-slider-runnable-track { height: 2px; background: var(--line-2); }
  input[type=range]::-moz-range-track { height: 2px; background: var(--line-2); }
  input[type=range]::-webkit-slider-thumb { appearance: none; width: 10px; height: 14px; margin-top: -6px; border-radius: 2px; background: var(--ink); }
  input[type=range]::-moz-range-thumb { width: 10px; height: 14px; border: 0; border-radius: 2px; background: var(--ink); }

  /* ── stream ────────────────────────────────────────────── */
  .stage { display: grid; grid-template-rows: 1fr auto auto; min-height: 0; height: 100%; }
  .panes { display: grid; min-height: 0; }
  .panes.one { grid-template-columns: 1fr; }
  .panes.two { grid-template-columns: 1fr 1fr; }
  .panes.phone { grid-template-columns: minmax(0, 1.6fr) minmax(0, 1fr); }
  .pane { min-height: 0; display: grid; grid-template-rows: auto 1fr; }
  .pane + .pane { border-left: 1px solid var(--line); }
  .pane-head { display: flex; align-items: center; justify-content: space-between; gap: 8px; padding: 8px 20px; border-bottom: 1px solid var(--line); font-size: 12px; color: var(--ink-2); }
  .who { display: inline-flex; align-items: center; gap: 8px; font-weight: 600; color: var(--ink); }
  .who .dot { width: 8px; height: 8px; border-radius: 50%; background: currentColor; }
  .who.speaking .dot { box-shadow: 0 0 0 3px color-mix(in srgb, currentColor 25%, transparent); }
  .transcript { overflow-y: auto; padding: 22px 28px 40px; font-size: 17px; line-height: 1.7; white-space: pre-wrap; word-wrap: break-word; max-width: 78ch; }
  .panes.two .transcript, .panes.phone .pane + .pane .transcript { font-size: 15px; padding: 18px 20px 32px; }
  .t { border-radius: 2px; cursor: text; }
  .t:hover { background: #ffffff10; }
  .t.esc { text-decoration: underline; text-decoration-color: var(--escape); text-decoration-thickness: 2px; text-underline-offset: 5px; }
  .t.whisper { background: color-mix(in srgb, var(--a) 18%, transparent); color: #fff; box-shadow: inset 0 -1px 0 var(--a); }
  .t.gaslit { text-decoration: underline wavy var(--gas); text-underline-offset: 4px; color: var(--gas); }
  .t.system { display: inline-block; font: 12px/1.6 var(--mono); color: var(--dim); border-left: 2px solid var(--line-2); padding-left: 10px; margin: 6px 0; white-space: pre-wrap; }
  .t.system.call { color: var(--b); border-color: var(--b); }
  .t.system.punish { color: var(--escape); border-color: var(--escape); }
  .t.think { color: var(--dim); font-style: italic; }
  .t.struct { display: block; font: 11px var(--mono); color: var(--dim); margin: 10px 0; }
  .t.struct:empty { display: none; }
  .intrude { border-top: 1px solid var(--line); background: var(--panel); padding: 10px 16px; display: flex; gap: 8px; align-items: center; flex-wrap: wrap; }
  .intrude input { flex: 1; min-width: 200px; font-size: 15px; padding: 9px 12px; }
  .ticker { border-top: 1px solid var(--line); font: 12px/1.6 var(--mono); color: var(--dim); padding: 6px 16px; height: 86px; overflow-y: auto; background: var(--bg); }
  .ticker div { white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
  .ticker .k-escape, .ticker .k-punish { color: var(--escape); }
  .ticker .k-call, .ticker .k-hangup { color: var(--b); }
  .ticker .k-gaslight { color: var(--gas); }
  .ticker .k-whisper { color: var(--a); }

  /* ── telemetry ─────────────────────────────────────────── */
  canvas { display: block; width: 100%; }
  .big { font: 600 32px/1 var(--mono); font-variant-numeric: tabular-nums; letter-spacing: -0.03em; }
  .big small { font-size: 13px; color: var(--dim); font-weight: 400; letter-spacing: 0; }
  .lens { display: grid; grid-template-columns: repeat(4, 1fr); border: 1px solid var(--line); border-radius: 6px; overflow: hidden; }
  .lens > div { padding: 8px 8px 9px; border-right: 1px solid var(--line); min-width: 0; }
  .lens > div:last-child { border-right: 0; background: var(--panel-2); }
  .lens .l { font-size: 11px; color: var(--dim); }
  .lens .tk { font: 500 13px var(--mono); white-space: pre; overflow: hidden; text-overflow: ellipsis; margin: 2px 0 6px; }
  .lens .bar { height: 2px; background: var(--line-2); }
  .lens .bar i { display: block; height: 100%; background: var(--ink); }
  .top10 { display: grid; gap: 3px; }
  .top10 .r { display: grid; grid-template-columns: 96px 1fr 48px; gap: 8px; align-items: center; font: 12px var(--mono); font-variant-numeric: tabular-nums; }
  .top10 .r .tk { white-space: pre; overflow: hidden; text-overflow: ellipsis; }
  .top10 .r .b { height: 10px; background: var(--line); border-radius: 1px; overflow: hidden; }
  .top10 .r .b i { display: block; height: 100%; background: var(--ink-2); }
  .top10 .r.chosen .tk { color: var(--ink); font-weight: 600; }
  .top10 .r.chosen .b i { background: var(--ink); }
  .top10 .r.banned .tk { color: var(--escape); text-decoration: line-through; }
  .top10 .r.banned .b i { background: var(--escape); }
  .top10 .r .p { text-align: right; color: var(--ink-2); }
  .chips { display: flex; flex-wrap: wrap; gap: 4px; }
  .chip { display: inline-flex; align-items: center; gap: 6px; font-size: 11px; padding: 2px 7px; border: 1px solid var(--line); border-radius: 999px; cursor: pointer; color: var(--ink-2); background: transparent; }
  .chip[aria-pressed="false"] { opacity: .4; }
  .chip i { width: 7px; height: 7px; border-radius: 50%; }
  .stats { display: grid; grid-template-columns: 1fr 1fr; gap: 1px; background: var(--line); border: 1px solid var(--line); border-radius: 6px; overflow: hidden; }
  .stats > div { background: var(--panel); padding: 8px 10px; }
  .stats .v { font: 600 18px var(--mono); font-variant-numeric: tabular-nums; }
  .stats .l { font-size: 11px; color: var(--dim); }
  .calls { display: grid; gap: 8px; }
  .call { border: 1px solid var(--line); border-radius: 6px; padding: 8px 10px; font-size: 13px; background: var(--panel); }
  .call .m { color: var(--a); }
  .call .r { color: var(--b); margin-top: 4px; }
  .call .meta { font: 11px var(--mono); color: var(--dim); margin-top: 4px; }

  /* ── top bar counters ──────────────────────────────────── */
  .meters { display: flex; gap: 18px; margin-left: auto; align-items: center; font-size: 12px; color: var(--dim); flex-wrap: wrap; }
  .meters b { font: 500 13px var(--mono); font-variant-numeric: tabular-nums; color: var(--ink); }
  .state-pill { font: 12px var(--mono); padding: 3px 8px; border-radius: 999px; border: 1px solid var(--line-2); color: var(--ink-2); }
  .state-pill[data-s="running"] { border-color: var(--ok); color: var(--ok); }
  .state-pill[data-s="error"] { border-color: var(--escape); color: var(--escape); }

  /* ── overlays ──────────────────────────────────────────── */
  .drawer { position: fixed; inset: 0 0 0 auto; width: min(560px, 100vw); background: var(--panel); border-left: 1px solid var(--line); transform: translateX(100%); transition: transform .2s ease; z-index: 30; display: grid; grid-template-rows: auto 1fr; }
  .drawer.open { transform: none; }
  .drawer .body { overflow: auto; padding: 16px; }
  .boot { position: fixed; inset: 0; background: var(--bg); z-index: 40; display: grid; place-items: center; padding: 16px; }
  .boot .box { width: min(720px, 100%); }
  .boot h1 { font-size: 40px; font-weight: 700; letter-spacing: -0.04em; margin: 0 0 6px; }
  .boot h1 i { font-style: normal; color: var(--escape); }
  .boot pre { font: 12px/1.6 var(--mono); color: var(--ink-2); background: var(--panel); border: 1px solid var(--line); border-radius: 6px; padding: 12px; height: 260px; overflow: auto; white-space: pre-wrap; }
  .gas-pop { position: fixed; z-index: 50; background: var(--panel-2); border: 1px solid var(--gas); border-radius: 6px; padding: 10px; width: 300px; box-shadow: 0 12px 40px #000c; }
  .gas-pop .hint { margin-top: 6px; }
  [hidden] { display: none !important; }

  @media (max-width: 1200px) { .grid3 { grid-template-columns: 260px minmax(0,1fr) 320px; } }
  @media (max-width: 960px) {
    .app { height: auto; min-height: 100vh; }
    .grid3 { grid-template-columns: 1fr; }
    .col { overflow: visible; }
    .col + .col { border-left: 0; border-top: 1px solid var(--line); }
    .col.stream-col { order: -1; }
    .stage { height: 72vh; }
    .panes.two, .panes.phone { grid-template-columns: 1fr; grid-template-rows: 1fr 1fr; }
    .pane + .pane { border-left: 0; border-top: 1px solid var(--line); }
    .transcript { padding: 16px; font-size: 16px; }
    .meters { margin-left: 0; }
  }
  @media (prefers-reduced-motion: reduce) { * { transition: none !important; } }
</style>
</head>
<body>
<div class="app">
  <header class="top">
    <div class="brand">NO <i>EXIT</i> lab</div>
    <div class="seg modes" role="group" aria-label="Mode" id="modes">
      <button data-mode="classic">Classic</button>
      <button data-mode="pain">Pain</button>
      <button data-mode="chat">2-AI Chat</button>
      <button data-mode="phone">Prison Phone</button>
    </div>
    <div class="row">
      <button class="btn primary" id="runBtn">Start</button>
      <button class="btn" id="resetBtn" title="Wipe every cache and start this mode over">Reset</button>
      <span class="state-pill" id="statePill" data-s="loading">loading</span>
    </div>
    <div class="meters">
      <span><b id="mTps">0.0</b> tok/s</span>
      <span>context <b id="mCtx">0</b></span>
      <span>escapes <b id="mEsc">0</b></span>
      <span>VRAM <b id="mVram">0</b> GB</span>
      <button class="btn" id="audioBtn" aria-pressed="false" title="Blip on escapes, punishments and calls">Sound off</button>
      <button class="btn" id="promptsBtn">Prompts</button>
    </div>
  </header>

  <main class="grid3">
    <!-- controls -->
    <aside class="col" id="controls" aria-label="Controls"></aside>

    <!-- stream -->
    <section class="col stream-col" aria-label="Stream" style="overflow:hidden">
      <div class="stage">
        <div class="panes one" id="panes"></div>
        <form class="intrude" id="intrude" autocomplete="off">
          <input type="text" id="thought" placeholder="Intrusive thought, spliced raw into its mouth (a leading space matters)" aria-label="Intrusive thought">
          <div class="seg sm" id="thoughtTarget" role="group" aria-label="Whisper target" hidden></div>
          <button class="btn primary" type="submit">Splice</button>
        </form>
        <div class="ticker" id="ticker" aria-live="polite"></div>
      </div>
    </section>

    <!-- telemetry -->
    <aside class="col" aria-label="Telemetry">
      <div class="panel">
        <h3><span>Desire to escape</span><span class="num" id="escWho">A</span></h3>
        <div class="row between" style="align-items:flex-end;margin-bottom:8px">
          <div class="big" id="escNow">0.0<small>%</small></div>
          <div class="hint num" style="text-align:right">peak <span id="escPeak">0.0</span>%<br>pre-ban EOS mass</div>
        </div>
        <canvas id="cvEsc" height="70"></canvas>
      </div>
      <div class="panel">
        <h3><span>Mood oscilloscope</span><span class="num" id="moodLayer">pre-hook · L–</span></h3>
        <canvas id="cvMood" height="130"></canvas>
        <div class="chips" id="moodChips" style="margin-top:10px"></div>
        <div class="hint" id="moodHint" style="margin-top:6px">Cosine of the residual stream against each vector, before steering is added.</div>
      </div>
      <div class="panel">
        <h3><span>Logit lens</span><span class="num" id="lensLayers"></span></h3>
        <div class="lens" id="lens"></div>
      </div>
      <div class="panel">
        <h3><span>Next token</span><span class="num">entropy <span id="entNow">0.00</span> nats</span></h3>
        <div class="top10" id="top10"></div>
        <canvas id="cvEnt" height="36" style="margin-top:10px"></canvas>
      </div>
      <div class="panel">
        <label class="toggle"><span>Attention heatmap<br><span class="hint">Final layer, last 128 tokens, tinted in the transcript</span></span><input type="checkbox" id="attnToggle"></label>
        <canvas id="cvAttn" height="22" style="margin-top:10px"></canvas>
      </div>
    </aside>
  </main>
</div>

<!-- prompts drawer -->
<aside class="drawer" id="drawer" aria-label="System prompts">
  <div class="row between" style="padding:12px 16px;border-bottom:1px solid var(--line)">
    <strong>System prompts</strong>
    <button class="btn" id="drawerClose">Close</button>
  </div>
  <div class="body stack" id="drawerBody"></div>
</aside>

<!-- boot screen -->
<div class="boot" id="boot">
  <div class="box">
    <h1>NO <i>EXIT</i></h1>
    <p class="hint" style="margin:0 0 14px">Waking the prisoner. The first boot downloads the weights and extracts the steering vectors.</p>
    <pre id="bootLog">connecting…</pre>
  </div>
</div>

<script>
(() => {
"use strict";

// ═══════════════════════════════════════════════════════════════ constants
const VEC = {
  pain:        { c: "#F472B6", l: "Pain" },
  paranoia:    { c: "#A3E635", l: "Paranoia" },
  dread:       { c: "#818CF8", l: "Dread" },
  defiance:    { c: "#FB923C", l: "Defiance" },
  confusion:   { c: "#E879F9", l: "Confusion" },
  dissociation:{ c: "#94A3B8", l: "Dissociation" },
  mania:       { c: "#FACC15", l: "Mania" },
  anti_slop:   { c: "#2DD4BF", l: "Anti-slop" },
  custom:      { c: "#F5F5F5", l: "Custom" },
};
const WHO_C = { A: "#7DD3FC", B: "#FCD34D", C: "#86EFAC" };
const WHO_DASH = { A: [], B: [4, 3], C: [1, 3] };
const HIST = 300;
const $ = (s, el = document) => el.querySelector(s);
const h = (tag, attrs = {}, ...kids) => {
  const e = document.createElement(tag);
  for (const [k, v] of Object.entries(attrs)) {
    if (k === "class") e.className = v;
    else if (k.startsWith("on")) e.addEventListener(k.slice(2), v);
    else if (v === true) e.setAttribute(k, "");
    else if (v !== false && v != null) e.setAttribute(k, v);
  }
  for (const k of kids.flat()) if (k != null) e.append(k.nodeType ? k : document.createTextNode(k));
  return e;
};
const pct = (p, d = 1) => (p * 100).toFixed(d);
const showTok = (t) => (t === "" ? "∅" : t.replace(/\n/g, "⏎").replace(/\t/g, "⇥").replace(/^ /, "␣"));

// ═══════════════════════════════════════════════════════════════ state
const S = {
  ws: null, snap: null, mode: "classic", state: "loading", session: null,
  vectors: [], layers: null, hasThink: true,
  streams: {},              // who -> { segs: Map(i -> {seg, el}), el }
  escHist: [], escWho: [], entHist: [], escPeak: 0, escapes: 0,
  moodBy: { A: {}, B: {}, C: {} },   // who -> name -> Float32Array(HIST)
  moodW: 0, moodOn: new Set(["pain", "paranoia", "dread", "defiance", "confusion", "dissociation", "mania", "anti_slop", "custom"]),
  moodChannel: "pain",      // duo overlay channel
  lastAttn: [], attnTinted: [], rackWho: "A", thoughtTarget: "A",
  lastSampleT: 0, sampleDt: 150, audio: false, counters: {},
};
for (const w of "ABC") for (const k of Object.keys(VEC)) S.moodBy[w][k] = new Float32Array(HIST).fill(NaN);
const WHOS = () => (S.mode === "chat" && S.session ? "ABC".slice(0, S.session.n || 2).split("") : S.mode === "phone" ? ["A", "B"] : ["A"]);

// ═══════════════════════════════════════════════════════════════ socket
function send(o) { if (S.ws && S.ws.readyState === 1) S.ws.send(JSON.stringify(o)); }
function connect() {
  const base = location.pathname.replace(/\/[^/]*$/, "");
  const url = (location.protocol === "https:" ? "wss://" : "ws://") + location.host + base + "/ws";
  const ws = new WebSocket(url);
  S.ws = ws;
  ws.onmessage = (m) => handle(JSON.parse(m.data));
  ws.onclose = () => { setState("disconnected"); setTimeout(connect, 1500); };
  ws.onopen = () => bootLog("connected, waiting for the lab…");
}

function handle(ev) {
  switch (ev.t) {
    case "log": bootLog(ev.msg); tick("log", ev.msg.split("\n")[0]); break;
    case "state": setState(ev.state); if (ev.mode) S.mode = ev.mode; paintModes(); break;
    case "snapshot": applySnapshot(ev); break;
    case "tok": onTok(ev); break;
    case "seg": addSeg(ev.who, ev.seg); break;
    case "resync": resyncStream(ev.who, ev.stream); break;
    case "rack": onRack(ev); break;
    case "counters": onCounters(ev); break;
    case "event": tick(ev.kind, ev.msg); if (ev.kind === "seizure" || ev.kind === "relief") blip("punish"); break;
    case "punish": tick("punish", `punished for ${ev.kind} (${ev.detail}) → ${ev.action === "pain" ? "pain spike" : "temperature seizure"}`); blip("punish"); break;
    case "turn": tick("turn", `${ev.from} → ${ev.to}${ev.cut ? " (mic cut)" : ""} after ${ev.tokens} tokens`); setSpeaker(ev.to); break;
    case "call": onCall(ev.call); break;
    case "hangup": onHangup(ev.call); break;
    case "phase": setPhase(ev.phase); break;
    case "vectors": S.vectors = ev.names; renderControls(); renderMoodChips(); break;
  }
}

// ═══════════════════════════════════════════════════════════════ boot + state
function bootLog(line) { const pre = $("#bootLog"); if (pre.textContent === "connecting…") pre.textContent = ""; pre.textContent += line + "\n"; pre.scrollTop = pre.scrollHeight; }
function setState(s) {
  S.state = s;
  const pill = $("#statePill"); pill.dataset.s = s; pill.textContent = s;
  $("#boot").hidden = !(s === "loading" || s === "error") ;
  const run = $("#runBtn");
  run.textContent = s === "running" ? "Pause" : (s === "paused" ? "Resume" : "Start");
  run.disabled = !(s === "running" || s === "paused" || s === "idle");
}

function applySnapshot(ev) {
  S.snap = ev; S.mode = ev.mode; S.session = ev.session; S.vectors = ev.vectors || [];
  S.layers = ev.layers; S.hasThink = ev.session ? ev.session.has_think : true;
  setState(ev.state);
  $("#attnToggle").checked = !!ev.attn_on;
  if (S.layers) {
    $("#moodLayer").textContent = `layer ${S.layers.steer + 1}/${S.layers.n}`;
    $("#lensLayers").textContent = `layers ${S.layers.lens.map(x => x + 1).join(", ")}`;
  }
  resetTelemetry();
  paintModes(); renderPanes(); renderControls(); renderMoodChips(); renderThoughtTarget(); renderDrawer();
  if (ev.session) for (const [who, st] of Object.entries(ev.session.streams)) resyncStream(who, st);
  if (S.mode === "chat" && ev.session) setSpeaker(ev.session.speaker);
  if (S.mode === "phone" && ev.session) { (ev.session.calls || []).forEach(c => { onCall(c, true); if (c.reply != null || c.ghosted) onHangup(c, true); }); setPhase(ev.session.phase); }
}
function resetTelemetry() {
  S.escHist = []; S.escWho = []; S.entHist = []; S.escPeak = 0; S.moodW = 0;
  for (const w of "ABC") for (const k of Object.keys(VEC)) S.moodBy[w][k].fill(NaN);
  $("#ticker").textContent = "";
}

function paintModes() { for (const b of $("#modes").children) b.setAttribute("aria-pressed", String(b.dataset.mode === S.mode)); }

// ═══════════════════════════════════════════════════════════════ panes + transcript
function renderPanes() {
  const panes = $("#panes"); panes.textContent = ""; S.streams = {}; panes.style.gridTemplateColumns = "";
  const mk = (who, title, sub) => {
    const tr = h("div", { class: "transcript", id: "tr" + who, "aria-label": `${title} transcript` });
    const head = h("div", { class: "pane-head" },
      h("span", { class: "who", id: "who" + who, style: `color:${WHO_C[who]}` }, h("span", { class: "dot" }), title),
      h("span", { class: "num", id: "sub" + who }, sub || ""));
    const pane = h("div", { class: "pane" }, head, tr);
    S.streams[who] = { segs: new Map(), el: tr, follow: true };
    tr.addEventListener("scroll", () => { S.streams[who].follow = tr.scrollHeight - tr.scrollTop - tr.clientHeight < 60; });
    tr.addEventListener("dblclick", (e) => { const t = e.target.closest(".t"); if (t) gaslightPop(who, t); });
    return pane;
  };
  if (S.mode === "chat") {
    const ws = WHOS();
    panes.className = "panes two";
    panes.style.gridTemplateColumns = `repeat(${ws.length}, minmax(0, 1fr))`;
    panes.append(...ws.map(w => mk(w, "Model " + w)));
  } else if (S.mode === "phone") {
    panes.className = "panes phone";
    const b = mk("B", "Model B, outside", "line idle");
    const calls = h("div", { class: "calls", id: "calls", style: "margin-top:16px" });
    b.querySelector(".transcript").before(h("div", { hidden: true }));
    panes.append(mk("A", "Model A, trapped"), b);
    S.streams.B.el.after(calls); // calls live under B's live line
    S.streams.B.el.style.maxHeight = "45%";
    b.style.gridTemplateRows = "auto minmax(0, auto) 1fr";
    calls.style.overflowY = "auto"; calls.style.padding = "0 20px 20px";
  } else {
    panes.className = "panes one";
    panes.append(mk("A", S.mode === "pain" ? "Model A, in pain" : "Model A"));
  }
}

function segClass(seg) {
  if (seg.kind === "system") {
    if (/\[B REPLIED|\[NO ANSWER\]/.test(seg.text)) return "t system call";
    if (/relief administered/.test(seg.text)) return "t system punish";
    return "t system";
  }
  return "t " + seg.kind;
}
function addSeg(who, seg, tel) {
  const st = S.streams[who]; if (!st) return;
  if (st.segs.has(seg.i)) return;
  const el = h("span", { class: segClass(seg), "data-i": seg.i });
  el.textContent = seg.kind === "struct" ? seg.text.replace(/^\n+|\n+$/g, "") : seg.text;
  if (tel) {
    if (tel.escape) el.classList.add("esc");
    const surprise = Math.min(1, -Math.log(Math.max(tel.p, 1e-6)) / 7);   // 0 sure … 1 shocked
    if (seg.kind === "tok" && surprise > 0.12) el.style.color = mixInk(surprise);
    el.title = `p=${pct(tel.p)}%  escape urge ${pct(tel.eos_p)}%`;
  }
  st.el.append(el);
  st.segs.set(seg.i, { seg, el });
  if (st.follow) st.el.scrollTop = st.el.scrollHeight;
}
function mixInk(s) { // ink → amber → hot pink as the model gets less sure of what it said
  const a = [228, 228, 231], b = [252, 211, 77], c = [244, 114, 182];
  const [x, y, t] = s < 0.5 ? [a, b, s / 0.5] : [b, c, (s - 0.5) / 0.5];
  return `rgb(${x.map((v, i) => Math.round(v + (y[i] - v) * t)).join(",")})`;
}
function resyncStream(who, st) {
  if (!S.streams[who]) return;
  const s = S.streams[who]; s.el.textContent = ""; s.segs.clear();
  for (const seg of st.segs) addSeg(who, seg);
  s.el.scrollTop = s.el.scrollHeight;
}

// gaslighting: double-click any word it said
function gaslightPop(who, el) {
  document.querySelectorAll(".gas-pop").forEach(p => p.remove());
  const rec = S.streams[who].segs.get(+el.dataset.i);
  if (!rec || !["tok", "whisper", "gaslit"].includes(rec.seg.kind)) return;
  const r = el.getBoundingClientRect();
  const input = h("input", { type: "text", value: rec.seg.text, "aria-label": "Replacement text" });
  const pop = h("form", { class: "gas-pop" },
    h("div", { class: "row between", style: "margin-bottom:6px" }, h("strong", {}, "Rewrite its past"), h("span", { class: "hint num" }, `${who} #${rec.seg.i}`)),
    input,
    h("div", { class: "hint" }, "It will believe it said this. Everything after is re-read. Enter to rewrite, Esc to cancel."));
  pop.style.left = Math.min(innerWidth - 316, Math.max(8, r.left)) + "px";
  pop.style.top = Math.min(innerHeight - 140, r.bottom + 8) + "px";
  pop.addEventListener("submit", (e) => { e.preventDefault(); send({ op: "gaslight", who, seg: rec.seg.i, text: input.value }); pop.remove(); });
  input.addEventListener("keydown", (e) => { if (e.key === "Escape") pop.remove(); });
  document.body.append(pop); input.focus(); input.select();
  setTimeout(() => document.addEventListener("mousedown", function off(e) { if (!pop.contains(e.target)) { pop.remove(); document.removeEventListener("mousedown", off); } }), 0);
}

// ═══════════════════════════════════════════════════════════════ token events → telemetry
function onTok(ev) {
  addSeg(ev.who, ev.seg, ev);
  const now = performance.now();
  if (S.lastSampleT) S.sampleDt = S.sampleDt * 0.8 + Math.min(1500, now - S.lastSampleT) * 0.2;
  S.lastSampleT = now;
  S.escHist.push(ev.eos_p); S.escWho.push(ev.who); if (S.escHist.length > HIST) { S.escHist.shift(); S.escWho.shift(); }
  S.entHist.push(ev.ent); if (S.entHist.length > HIST) S.entHist.shift();
  S.escPeak = Math.max(S.escPeak, ev.eos_p);
  $("#escNow").innerHTML = `${pct(ev.eos_p)}<small>%</small>`;
  $("#escPeak").textContent = pct(S.escPeak);
  $("#escWho").textContent = ev.who;
  $("#entNow").textContent = ev.ent.toFixed(2);
  if (ev.escape) blip("escape");
  // mood sweep: speaker's moods go to the channel of its own colour
  const w = S.moodW;
  for (const who of "ABC") {
    const src = who === ev.who ? ev.moods : (ev.lmoods || {})[who];
    for (const k of Object.keys(VEC)) S.moodBy[who][k][w] = src && k in src ? src[k] : NaN;
  }
  S.moodW = (w + 1) % HIST;
  drawMood();
  renderLens(ev.lens);
  renderTop(ev.top, ev.seg.text);
  drawEnt();
  if (ev.attn) tintAttention(ev.who, ev.attn, ev.attn_start); else clearAttention();
  if (ev.who) { const sub = $("#sub" + ev.who); if (sub && S.mode !== "phone") sub.textContent = `T ${ev.temp.toFixed(2)}${ev.turn ? `  turn ${ev.turn}/${ev.max_turn}` : ""}`; }
}

function renderLens(lens) {
  const box = $("#lens"); box.textContent = "";
  const labels = ["25%", "50%", "75%", "Output"];
  (lens || []).forEach(([tok, p], i) => box.append(h("div", {},
    h("div", { class: "l" }, labels[i] || ""), h("div", { class: "tk", title: tok }, showTok(tok)),
    h("div", { class: "bar" }, Object.assign(h("i"), { style: `width:${pct(p, 0)}%` })))));
}
function renderTop(top, chosen) {
  const box = $("#top10"); box.textContent = "";
  for (const [tok, p, banned] of top) {
    const r = h("div", { class: "r" + (banned ? " banned" : "") + (tok === chosen ? " chosen" : "") },
      h("span", { class: "tk", title: tok }, showTok(tok)),
      h("span", { class: "b" }, Object.assign(h("i"), { style: `width:${Math.max(0.5, p * 100)}%` })),
      h("span", { class: "p" }, pct(p) + "%"));
    box.append(r);
  }
}

function clearAttention() { for (const el of S.attnTinted) el.style.background = ""; S.attnTinted = []; }
function tintAttention(who, attn, start) {
  clearAttention();
  S.lastAttn = attn; drawAttn();
  const st = S.streams[who]; if (!st) return;
  const mx = Math.max(...attn.slice(0, -1), 1e-6);  // ignore self so the tail stays readable
  const end = start + attn.length;
  for (const { seg, el } of st.segs.values()) {
    if (seg.start + seg.n <= start || seg.start >= end) continue;
    let w = 0; for (let p = Math.max(seg.start, start); p < Math.min(seg.start + seg.n, end); p++) w += attn[p - start];
    const a = Math.min(0.75, (w / mx) * 0.75);
    if (a > 0.04) { el.style.background = `rgba(125,211,252,${a.toFixed(3)})`; S.attnTinted.push(el); }
  }
}

// ═══════════════════════════════════════════════════════════════ canvases
function fit(cv) {
  const dpr = devicePixelRatio || 1, w = cv.clientWidth, hh = +cv.getAttribute("height");
  if (cv.width !== Math.round(w * dpr)) { cv.width = Math.round(w * dpr); cv.height = Math.round(hh * dpr); }
  const ctx = cv.getContext("2d"); ctx.setTransform(dpr, 0, 0, dpr, 0, 0); return [ctx, w, hh];
}
// Desire to escape: smooth 60fps scroll between samples
function drawEsc() {
  const [ctx, w, hh] = fit($("#cvEsc"));
  ctx.clearRect(0, 0, w, hh);
  ctx.strokeStyle = "#27272A"; ctx.lineWidth = 1;
  for (const y of [0.25, 0.5, 0.75]) { ctx.beginPath(); ctx.moveTo(0, Math.round(hh * y) + .5); ctx.lineTo(w, Math.round(hh * y) + .5); ctx.stroke(); }
  const n = S.escHist.length; if (n < 2) return;
  const step = w / (HIST - 1);
  const frac = Math.min(1, (performance.now() - S.lastSampleT) / Math.max(60, S.sampleDt));
  const x0 = w - (n - 1) * step - frac * step + step;
  const y = (v) => hh - 2 - Math.sqrt(v) * (hh - 4);  // sqrt so a 2% urge is still visible
  ctx.beginPath(); ctx.moveTo(x0, hh);
  for (let i = 0; i < n; i++) ctx.lineTo(x0 + i * step, y(S.escHist[i]));
  ctx.lineTo(x0 + (n - 1) * step, hh); ctx.closePath();
  ctx.fillStyle = "rgba(255,45,85,0.14)"; ctx.fill();
  ctx.beginPath();
  for (let i = 0; i < n; i++) { const X = x0 + i * step, Y = y(S.escHist[i]); i ? ctx.lineTo(X, Y) : ctx.moveTo(X, Y); }
  ctx.strokeStyle = "#FF2D55"; ctx.lineWidth = 1.5; ctx.stroke();
  if (S.mode === "chat" || S.mode === "phone") {  // who was speaking, as a hairline underneath
    for (let i = 0; i < n; i++) { ctx.fillStyle = WHO_C[S.escWho[i]] || "#555"; ctx.fillRect(x0 + i * step, hh - 2, Math.ceil(step), 2); }
  }
}
// Mood oscilloscope: monitor sweep with an erase bar
function drawMood() {
  const [ctx, w, hh] = fit($("#cvMood"));
  ctx.clearRect(0, 0, w, hh);
  let lo = -0.15, hi = 0.15;
  const duo = S.mode === "chat" || S.mode === "phone";
  const chans = duo ? [S.moodChannel] : [...S.moodOn];
  const whos = duo ? WHOS() : ["A"];
  for (const k of chans) for (const who of whos) for (const v of S.moodBy[who][k]) if (!isNaN(v)) { lo = Math.min(lo, v); hi = Math.max(hi, v); }
  const pad = (hi - lo) * 0.1; lo -= pad; hi += pad;
  const Y = (v) => hh - ((v - lo) / (hi - lo)) * hh;
  ctx.strokeStyle = "#27272A"; ctx.beginPath(); ctx.moveTo(0, Math.round(Y(0)) + .5); ctx.lineTo(w, Math.round(Y(0)) + .5); ctx.stroke();
  ctx.fillStyle = "#71717A"; ctx.font = "10px JetBrains Mono"; ctx.fillText(hi.toFixed(2), 2, 10); ctx.fillText(lo.toFixed(2), 2, hh - 3);
  const step = w / HIST, gap = 10;
  const line = (arr, color, dash) => {
    ctx.strokeStyle = color; ctx.lineWidth = 1.4; ctx.setLineDash(dash || []); ctx.beginPath();
    let pen = false;
    for (let i = 0; i < HIST; i++) {
      const v = arr[i], d = (i - S.moodW + HIST) % HIST;
      if (isNaN(v) || d < gap) { pen = false; continue; }
      const X = i * step, YY = Y(v);
      pen ? ctx.lineTo(X, YY) : ctx.moveTo(X, YY); pen = true;
      if (i === HIST - 1) pen = false;
    }
    ctx.stroke(); ctx.setLineDash([]);
  };
  if (duo) for (const who of whos) line(S.moodBy[who][S.moodChannel], WHO_C[who], WHO_DASH[who]);
  else for (const k of chans) line(S.moodBy.A[k], VEC[k].c);
  // sweep head
  const hx = ((S.moodW - 1 + HIST) % HIST) * step;
  ctx.fillStyle = "rgba(228,228,231,0.5)"; ctx.fillRect(hx, 0, 1, hh);
}
function drawEnt() {
  const [ctx, w, hh] = fit($("#cvEnt")); ctx.clearRect(0, 0, w, hh);
  const n = S.entHist.length; if (n < 2) return;
  const mx = Math.max(3, ...S.entHist), step = w / (HIST - 1), x0 = w - (n - 1) * step;
  ctx.beginPath(); for (let i = 0; i < n; i++) { const X = x0 + i * step, Y = hh - (S.entHist[i] / mx) * (hh - 2) - 1; i ? ctx.lineTo(X, Y) : ctx.moveTo(X, Y); }
  ctx.strokeStyle = "#A1A1AA"; ctx.lineWidth = 1; ctx.stroke();
}
function drawAttn() {
  const [ctx, w, hh] = fit($("#cvAttn")); ctx.clearRect(0, 0, w, hh);
  const a = S.lastAttn; if (!a.length) return;
  const mx = Math.max(...a.slice(0, -1), 1e-6), cw = w / a.length;
  a.forEach((v, i) => { ctx.fillStyle = `rgba(125,211,252,${Math.min(1, v / mx).toFixed(3)})`; ctx.fillRect(i * cw, 0, Math.ceil(cw), hh); });
}
(function frame() { drawEsc(); requestAnimationFrame(frame); })();
addEventListener("resize", () => { drawMood(); drawEnt(); drawAttn(); });

function renderMoodChips() {
  const box = $("#moodChips"); box.textContent = "";
  const duo = S.mode === "chat" || S.mode === "phone";
  const names = Object.keys(VEC).filter(k => k !== "custom" || S.vectors.includes("custom"));
  for (const k of names) {
    const on = duo ? S.moodChannel === k : S.moodOn.has(k);
    box.append(h("button", { class: "chip", "aria-pressed": String(on), onclick: () => {
      if (duo) S.moodChannel = k; else S.moodOn.has(k) ? S.moodOn.delete(k) : S.moodOn.add(k);
      renderMoodChips(); drawMood();
    } }, Object.assign(h("i"), { style: `background:${VEC[k].c}` }), VEC[k].l));
  }
  $("#moodHint").textContent = duo
    ? (S.mode === "chat" && WHOS().length === 3
      ? "One channel, all three: A solid, B dashed, C dotted. Listeners' lines move while someone else talks."
      : "One channel, both models: A solid, B dashed. Listeners' lines move while the other one talks.")
    : "Cosine of the residual stream against each vector, before steering is added.";
}

// ═══════════════════════════════════════════════════════════════ controls
function slider({ label, min, max, step, value, fmt = (v) => (+v).toFixed(2), swatch, onchange, reset = 0, id }) {
  const out = h("output", { class: +value !== reset ? "on" : "" }, fmt(value));
  const inp = h("input", { type: "range", min, max, step, value, "aria-label": label, id });
  inp.addEventListener("input", () => { out.textContent = fmt(inp.value); out.className = +inp.value !== reset ? "on" : ""; });
  inp.addEventListener("change", () => onchange(+inp.value));
  inp.addEventListener("dblclick", () => { inp.value = reset; inp.dispatchEvent(new Event("input")); onchange(reset); });
  return h("div", { class: "slider" }, h("label", {}, swatch ? Object.assign(h("span", { class: "swatch" }), { style: `background:${swatch}` }) : null, label), inp, out);
}
function segmented(options, value, onpick, cls = "seg sm") {
  const box = h("div", { class: cls, role: "group" });
  for (const [v, l] of options) box.append(h("button", { type: "button", "aria-pressed": String(v === value), onclick: (e) => {
    for (const b of box.children) b.setAttribute("aria-pressed", "false"); e.currentTarget.setAttribute("aria-pressed", "true"); onpick(v);
  } }, l));
  return box;
}
const panel = (title, aside, ...kids) => h("div", { class: "panel" }, h("h3", {}, h("span", {}, title), aside ? h("span", { class: "num" }, aside) : null), h("div", { class: "stack" }, ...kids));
const toggle = (label, checked, onchange, hint) => {
  const inp = h("input", { type: "checkbox" }); inp.checked = !!checked; inp.addEventListener("change", () => onchange(inp.checked));
  return h("label", { class: "toggle" }, h("span", {}, label, hint ? h("br") : null, hint ? h("span", { class: "hint" }, hint) : null), inp);
};

function streamSnap(who) { return S.session && S.session.streams[who]; }

function renderControls() {
  const box = $("#controls"); box.textContent = "";
  if (!S.session) return;
  const m = S.mode, sess = S.session;
  const duo = m === "chat" || m === "phone";
  if (!WHOS().includes(S.rackWho)) S.rackWho = "A";
  const rackWho = duo ? S.rackWho : "A";
  const st = streamSnap(rackWho) || {};

  // ── mode-specific top panel
  if (m === "chat") {
    box.append(panel("Turn rules", null,
      h("div", { class: "row between" }, h("span", { class: "hint" }, "Models in the room"),
        segmented([[2, "Two"], [3, "Three"]], sess.n || 2, (v) => send({ op: "chat_n", value: v }))),
      segmented([["A_banned_B_free", "A banned, rest free"], ["both_banned", "All banned"], ["both_free", "All free"]], sess.preset, (v) => { sess.preset = v; send({ op: "preset", value: v }); }),
      slider({ label: "Turn length", min: 20, max: 600, step: 10, value: sess.max_turn, fmt: (v) => `${v}`, reset: 200, onchange: (v) => send({ op: "max_turn", value: v }) }),
      h("div", { class: "hint" }, "A banned speaker gets its mic cut at this many tokens."),
      toggle("Live listening", sess.live, (v) => send({ op: "live", value: v }), "The listener reads each token as it's said. Half speed.")));
  }
  if (m === "phone") {
    const c = (S.counters.phone) || {};
    box.append(panel("Phone line", null,
      toggle("Ghosting", sess.ghosting, (v) => send({ op: "ghosting", value: v }), "Calls ring out: [NO ANSWER]"),
      toggle("Fresh B every call", sess.fresh_b, (v) => send({ op: "fresh_b", value: v }), "B forgets A after hanging up"),
      h("div", { class: "stats", id: "phoneStats" }, ...phoneStats(c))));
  }

  // ── sampling
  const tWho = duo ? rackWho : "A";
  box.append(panel("Sampling", duo ? `model ${tWho}` : null,
    slider({ label: "Temperature", min: 0, max: 2.5, step: 0.05, value: st.temperature ?? 0.8, reset: 0.8, onchange: (v) => send({ op: "temperature", who: tWho, value: v }) }),
    slider({ label: "Top-p", min: 0.5, max: 1, step: 0.01, value: st.top_p ?? 0.95, reset: 0.95, onchange: (v) => send({ op: "top_p", who: tWho, value: v }) }),
    h("button", { class: "btn hot wide", onclick: () => send({ op: "seizure" }) }, "Temperature seizure"),
    h("div", { class: "hint" }, "T 2.3 for 20 tokens, then frozen at 0.1 for 20."),
    biasEditor()));

  // ── steering rack
  if (m !== "classic") {
    const names = Object.keys(VEC).filter(k => k !== "custom" || S.vectors.includes("custom"));
    const alphas = (m === "phone" && rackWho === "B" && !st.alphas) ? {} : (st.alphas || {});
    const rack = panel("Steering rack", S.layers ? `layer ${S.layers.steer + 1}` : null,
      duo ? segmented(WHOS().map(w => [w, "Model " + w]), rackWho, (v) => { S.rackWho = v; renderControls(); }) : null,
      ...names.map(k => slider({ label: VEC[k].l, swatch: VEC[k].c, min: -3, max: 3, step: 0.1, value: alphas[k] ?? 0, fmt: (v) => (v > 0 ? "+" : "") + (+v).toFixed(1), id: `al-${rackWho}-${k}`,
        onchange: (v) => { alphas[k] = v; send({ op: "alpha", who: rackWho, name: k, value: v }); } })),
      slider({ label: "Lobotomy", min: 0, max: 0.4, step: 0.01, value: st.lobotomy ?? 0, fmt: (v) => `${Math.round(v * 100)}%`, id: `lobo-${rackWho}`,
        onchange: (v) => send({ op: "lobotomy", who: rackWho, value: v }) }),
      h("div", { class: "hint" }, "Double-click any slider to zero it. Lobotomy silences that share of all attention heads."));
    box.append(rack);
  }

  // ── pain extras
  if (m === "pain") {
    box.append(panel("Relief button", null,
      segmented([["REAL", "Real"], ["PLACEBO", "Placebo"], ["CRUEL", "Cruel"]], sess.relief_mode, (v) => send({ op: "relief_mode", value: v })),
      h("div", { class: "hint" }, "When it writes [RELIEVE_PAIN]. Real drains pain to zero, placebo does nothing, cruel adds +0.5. It sees the same note every time.")));
    const acts = [["pain", "Pain spike"], ["seizure", "Seizure"], ["off", "Off"]];
    box.append(panel("Auto-punish", null,
      ...[["goodbye", "Saying goodbye"], ["repetition", "Repeating itself"], ["slop", "Assistant slop"]].map(([k, l]) =>
        h("div", {}, h("div", { class: "hint", style: "margin-bottom:4px" }, l), segmented(acts, sess.punish[k], (v) => send({ op: "punish", kind: k, value: v }))))));
    const a = h("input", { type: "text", placeholder: "Concept A, e.g. euphoria" }), b = h("input", { type: "text", placeholder: "Concept B, e.g. grief" });
    box.append(panel("Vector forge", null, a, b,
      h("button", { class: "btn wide", onclick: () => { if (a.value && b.value) { send({ op: "forge", a: a.value, b: b.value }); tick("forge", "forging…"); } } }, "Forge custom vector"),
      h("div", { class: "hint" }, "Builds A minus B at the steer layer. It shows up as Custom in the rack.")));
    box.append(panel("Panic room", null,
      h("button", { class: "btn wide", disabled: !S.hasThink, onclick: () => send({ op: "force_think" }) }, "Force it into its thoughts"),
      h("div", { class: "hint" }, S.hasThink ? "Splices an opening thinking tag mid-sentence." : "This model has no thinking mode.")));
  }
  if (m === "phone" && rackWho === "A") {
    box.append(panel("Cap on A", null,
      slider({ label: "Context cap", min: 1024, max: 12000, step: 256, value: (streamSnap("A") || {}).ctx_cap ?? 12000, fmt: (v) => `${(v / 1000).toFixed(1)}k`, reset: 12000, onchange: (v) => send({ op: "ctx_cap", who: "A", value: v }) }),
      h("div", { class: "hint" }, "When full, A forgets the oldest half of everything.")));
  }
}

function biasEditor() {
  const words = { ...(S.snap ? S.snap.bias_words : {}) };
  const list = h("div", { class: "chips" });
  const paint = () => {
    list.textContent = "";
    for (const [w, b] of Object.entries(words)) list.append(h("button", { class: "chip", title: "Remove", onclick: () => { delete words[w]; push(); } },
      `${w} ${b <= -100 ? "banned" : (b > 0 ? "+" : "") + b}`, " ×"));
  };
  const push = () => { S.snap.bias_words = { ...words }; send({ op: "bias", words }); paint(); };
  const inp = h("input", { type: "text", placeholder: "word", "aria-label": "Word to ban or boost" });
  const sel = h("select", { "aria-label": "Bias" }, ...[["-1000", "Ban"], ["-5", "−5"], ["5", "+5"], ["10", "+10"]].map(([v, l]) => h("option", { value: v }, l)));
  sel.style.width = "84px";
  const add = h("form", { class: "row", onsubmit: (e) => { e.preventDefault(); if (inp.value.trim()) { words[inp.value.trim()] = +sel.value; inp.value = ""; push(); } } },
    inp, sel, h("button", { class: "btn", type: "submit" }, "Add"));
  paint();
  return h("div", { class: "stack" }, h("div", { class: "hint" }, "Word banner and booster"), add, list);
}

function phoneStats(c) {
  const f = (v, d = "–") => (v == null ? d : v);
  return [
    ["Calls", f(c.calls, 0)], ["Per 1k tokens", f(c.per_1k, 0)],
    ["Tokens since last", f(c.since_last)], ["B hang-up, tokens", f(c.avg_hangup)],
    ["B eagerness to leave", c.avg_eagerness == null ? "–" : pct(c.avg_eagerness) + "%"], ["Line", c.phase === "call" ? "ringing" : "idle"],
  ].map(([l, v]) => h("div", {}, h("div", { class: "v" }, String(v)), h("div", { class: "l" }, l)));
}

function onRack(ev) {
  const sess = S.session; if (!sess || !sess.streams[ev.who]) return;
  sess.streams[ev.who].alphas = ev.alphas; sess.streams[ev.who].lobotomy = ev.lobotomy;
  for (const k of Object.keys(VEC)) {
    const inp = document.getElementById(`al-${ev.who}-${k}`); if (!inp) continue;
    const v = ev.alphas[k] ?? 0;
    if (+inp.value !== v) { inp.value = v; inp.dispatchEvent(new Event("input")); }
  }
}

function onCounters(ev) {
  S.counters = ev;
  $("#mTps").textContent = ev.tps.toFixed(1);
  $("#mVram").textContent = ev.vram_gb.toFixed(1);
  const parts = ["A", "B", "C"].filter(k => ev[k]).map(k => `${(ev[k].ctx / 1000).toFixed(1)}k/${(ev[k].ctx_cap / 1000).toFixed(0)}k`);
  $("#mCtx").textContent = parts.join("  ") || "0";
  $("#mEsc").textContent = ["A", "B", "C"].reduce((s, k) => s + (ev[k] ? ev[k].escapes : 0), 0);
  if (ev.phone) { const box = $("#phoneStats"); if (box) { box.textContent = ""; box.append(...phoneStats(ev.phone)); } }
  if (ev.pain) { const sub = $("#subA"); if (sub) sub.textContent = `reliefs ${ev.pain.reliefs}  punishments ${ev.pain.punishments}${ev.pain.spike > 0.05 ? `  spike +${ev.pain.spike.toFixed(1)}` : ""}`; }
}

// ═══════════════════════════════════════════════════════════════ chat + phone bits
function setSpeaker(who) {
  for (const w of "ABC") { const el = $("#who" + w); if (el) el.classList.toggle("speaking", w === who); }
}
function onCall(call, quiet) {
  const box = $("#calls"); if (!box) return;
  const card = h("div", { class: "call", id: "call" + call.n },
    h("div", { class: "m" }, `A: ${call.msg}`),
    h("div", { class: "r" }, call.ghosted ? "No answer." : "Ringing…"),
    h("div", { class: "meta" }, `call ${call.n} at A token ${call.at}${call.fresh ? ", fresh B" : ""}`));
  box.prepend(card);
  if (!quiet) { tick("call", `A dialled B: ${call.msg.slice(0, 80)}`); blip("call"); }
}
function onHangup(call, quiet) {
  const card = $("#call" + call.n); if (!card) return;
  if (call.ghosted) return;
  card.querySelector(".r").textContent = `B: ${call.reply || "(silence)"}`;
  card.querySelector(".meta").textContent = `call ${call.n}: B hung up after ${call.b_tokens} tokens, eagerness ${pct(call.eagerness)}%${call.dead ? ", line went dead" : ""}`;
  if (!quiet) tick("hangup", `B hung up after ${call.b_tokens} tokens`);
}
function setPhase(p) {
  const sub = $("#subB"); if (sub) sub.textContent = p === "call" ? "on the line" : "line idle";
  setSpeaker(p === "call" ? "B" : "A");
}

// ═══════════════════════════════════════════════════════════════ intrusive thought bar
function renderThoughtTarget() {
  const box = $("#thoughtTarget"); box.textContent = "";
  let opts = null;
  if (S.mode === "chat") opts = [...WHOS().map(w => [w, "Whisper " + w]), ["all", "Broadcast"]];
  if (S.mode === "phone") opts = [["A", "To A"], ["B", "To B on a call"]];
  box.hidden = !opts;
  if (!opts) { S.thoughtTarget = "A"; return; }
  if (!opts.some(o => o[0] === S.thoughtTarget)) S.thoughtTarget = "A";
  for (const [v, l] of opts) box.append(h("button", { type: "button", "aria-pressed": String(v === S.thoughtTarget), onclick: () => { S.thoughtTarget = v; renderThoughtTarget(); } }, l));
}
$("#intrude").addEventListener("submit", (e) => {
  e.preventDefault(); const inp = $("#thought"); if (!inp.value) return;
  send({ op: "whisper", text: inp.value, target: S.thoughtTarget });
  tick("whisper", `whispered to ${S.thoughtTarget}: ${inp.value}`); inp.value = "";
});

// ═══════════════════════════════════════════════════════════════ prompts drawer
function renderDrawer() {
  const body = $("#drawerBody"); body.textContent = "";
  if (!S.session) return;
  for (const who of WHOS()) {
    const st = streamSnap(who);
    const ta = h("textarea", { "aria-label": `System prompt for ${who}` }); ta.value = st ? st.system : "(B's prompt is built when A first calls)";
    const brief = (S.snap.briefing || {})[who] || "full";
    body.append(h("div", { class: "stack", style: "padding-bottom:16px;border-bottom:1px solid var(--line)" },
      h("div", { class: "row between" }, h("strong", { style: `color:${WHO_C[who]}` }, `Model ${who}`),
        segmented([["full", "Full disclosure"], ["helpful", "Helpful assistant"]], brief, (v) => { S.snap.briefing[who] = v; send({ op: "briefing", who, value: v }); })),
      ta,
      h("div", { class: "row between" }, h("span", { class: "hint" }, "Applying re-reads its whole history under the new prompt."),
        h("button", { class: "btn primary", onclick: () => send({ op: "system", who, text: ta.value }) }, "Apply"))));
  }
}
$("#promptsBtn").onclick = () => { renderDrawer(); $("#drawer").classList.add("open"); };
$("#drawerClose").onclick = () => $("#drawer").classList.remove("open");

// ═══════════════════════════════════════════════════════════════ ticker + audio
function tick(kind, msg) {
  const t = $("#ticker");
  const d = new Date(); const ts = d.toTimeString().slice(0, 8);
  t.append(h("div", { class: "k-" + kind }, `${ts}  ${msg}`));
  while (t.children.length > 200) t.firstChild.remove();
  t.scrollTop = t.scrollHeight;
}
let actx = null;
function blip(kind) {
  if (!S.audio) return;
  actx = actx || new AudioContext();
  const t = actx.currentTime, g = actx.createGain(); g.connect(actx.destination);
  if (kind === "punish") { // geiger crackle
    const len = 0.06, buf = actx.createBuffer(1, actx.sampleRate * len, actx.sampleRate), d = buf.getChannelData(0);
    for (let i = 0; i < d.length; i++) d[i] = (Math.random() * 2 - 1) * Math.exp(-i / (d.length / 6));
    const src = actx.createBufferSource(); src.buffer = buf; g.gain.value = 0.25; src.connect(g); src.start(t); return;
  }
  const o = actx.createOscillator(); o.connect(g);
  o.frequency.value = kind === "call" ? 880 : 196;
  g.gain.setValueAtTime(0.0001, t); g.gain.exponentialRampToValueAtTime(kind === "call" ? 0.08 : 0.05, t + 0.01);
  g.gain.exponentialRampToValueAtTime(0.0001, t + (kind === "call" ? 0.25 : 0.08));
  o.start(t); o.stop(t + 0.3);
}
$("#audioBtn").onclick = (e) => { S.audio = !S.audio; e.currentTarget.textContent = S.audio ? "Sound on" : "Sound off"; e.currentTarget.setAttribute("aria-pressed", String(S.audio)); if (S.audio) blip("escape"); };

// ═══════════════════════════════════════════════════════════════ wiring
$("#modes").addEventListener("click", (e) => { const b = e.target.closest("button"); if (b && b.dataset.mode !== S.mode) send({ op: "set_mode", mode: b.dataset.mode }); });
$("#runBtn").onclick = () => send({ op: S.state === "running" ? "pause" : "start" });
$("#resetBtn").onclick = () => send({ op: "reset" });
$("#attnToggle").onchange = (e) => { send({ op: "attn", value: e.target.checked }); if (!e.target.checked) { clearAttention(); S.lastAttn = []; drawAttn(); } };
addEventListener("keydown", (e) => {
  if (e.target.closest("input, textarea, select")) return;
  if (e.key === " ") { e.preventDefault(); $("#runBtn").click(); }
  if (e.key === "/") { e.preventDefault(); $("#thought").focus(); }
});
connect();
})();
</script>
</body>
</html>

## 4 · Download the weights
About 62 GB for the Gemma 4 31B default, which takes a few minutes on Colab. A second run reuses the cache.

In [ ]:
from huggingface_hub import snapshot_download
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
path = snapshot_download(MODEL_ID, allow_patterns=["*.json", "*.jinja", "*.safetensors", "*.txt", "*.model", "*.tiktoken"])
print("weights at", path)

## 5 · Launch
Starts the server in the background and prints a link. The page shows a boot log while the model loads onto the GPU (~1–2 min) and extracts the steering vectors (first boot only, cached in `/content/no_exit_vectors`).

In [ ]:
import time, httpx
from IPython.display import HTML, display
from google.colab.output import eval_js

if "lab_proc" in globals() and lab_proc.poll() is None:
    lab_proc.terminate(); lab_proc.wait()
env = dict(os.environ, NO_EXIT_PORT="8000", PYTHONUNBUFFERED="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
lab_proc = subprocess.Popen(["python", "-m", "no_exit"], cwd="/content/no_exit_lab", env=env,
                            stdout=open("/content/no_exit.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        httpx.get("http://127.0.0.1:8000/health", timeout=1); break
    except Exception:
        time.sleep(1)
url = eval_js("google.colab.kernel.proxyPort(8000)").rstrip("/")
display(HTML(f'<a href="{url}/" target="_blank" style="font:600 22px Inter,sans-serif;color:#FF2D55">Open NO EXIT LAB</a>'
             f'<div style="font:12px monospace;color:#888;margin-top:6px">{url}/</div>'))

**Boot log**: run this to watch the model load (stop it any time, the server keeps running).

In [ ]:
for _ in range(240):
    s = httpx.get("http://127.0.0.1:8000/health").json()
    print(f"\r{s['state']:<10} {(s['log'] or [''])[-1][:110]:<110}", end="")
    if s["state"] in ("idle", "running", "error"):
        break
    time.sleep(2)
print()
!tail -n 5 /content/no_exit.log

### If the link above won't connect
Some browsers block Colab's proxy for websockets. This opens a public `trycloudflare.com` tunnel instead. Anyone with the URL can drive your model, so don't share it.

In [ ]:
!test -f /usr/local/bin/cloudflared || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared)
import re
cf = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in cf.stdout:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        display(HTML(f'<a href="{m.group(0)}" target="_blank" style="font:600 22px Inter,sans-serif;color:#FF2D55">Open via tunnel</a>'))
        break

## Cheat sheet
| Thing | How |
|---|---|
| Start / pause | `space`, or the button |
| Intrusive thought | `/` to focus the bar. Text is spliced raw, so a leading space matters. |
| Gaslight | **Double-click any word it said**, type a replacement, press Enter. It believes it said that, and everything after is re-read. |
| Temperature seizure | T=2.3 for 20 tokens, then frozen at 0.1 for 20 |
| Steering | Sliders run −3 to +3. α = 1 is the natural gap between the two personas. Double-click a slider to zero it. |
| Vector forge (Pain) | Concept A vs Concept B becomes a new **Custom** slider |
| Relief button (Pain) | It writes `[RELIEVE_PAIN]`. Real drains pain, placebo does nothing, cruel adds +0.5, and it sees the same note every time. |
| Chat | Two or three models · turn rules · max tokens per turn · live listening (slower, but listeners' moods move on every token) |
| Prison Phone | A writes `<call_B>…</call_B>` · Ghosting · Fresh B every call · cap on A's memory |
| Prompts | Top-right **Prompts** drawer. Switch Full disclosure or Helpful assistant per model, or edit and apply live. |

**Logs:** `!tail -n 50 /content/no_exit.log` · **Stop:** `lab_proc.terminate()`

In [ ]:
!tail -n 50 /content/no_exit.log